# Chapter 3 -- Tools as Interfaces (Ollama Local)

Work through this notebook **after reading** `notes/ch03-tools-as-interfaces.md`. Chapter 2 built the loop and treated the tools inside it as given. This notebook is about the tools themselves -- and it builds every schema in the industry-standard **OpenAI tool format** (`{"type": "function", "function": {...}}` with `parameters`), running against your real local Ollama model.

The chapter's one-sentence thesis, which every exercise below is a consequence of: **a tool schema is a user interface whose user is a model with exactly one shot at reading it correctly.** No README, no follow-up question, no colleague at the next desk -- the name, the parameter names, and the description string are the entire specification.

Six exercises have a stub to fill in: **three schema rewrites** (Section 2), a **truncate-with-cursor return value** (Section 4), an **instructional error validator** (Section 5), and a **destructive-tool approval gate** (Section 7).

Each exercise is followed by two cells. First an `assert` cell that grades your work **entirely offline** -- deterministic, so a flaky local model can never mark a correct answer wrong. Then a **Live Demo** cell that puts what you just wrote in front of the real model, so you can watch what that specific fix buys you: the model choosing between your bad and good schemas, paging through your continuation handle, repairing itself from your error message, and hitting your approval gate. Those demo cells need `ollama serve` running; the assert cells do not.

The last two sections then run your *bad* tool set and your *good* tool set against the real model back to back, so you can watch Section 9's selection-accuracy claim happen on your own machine rather than take it on faith.

## One-time local setup

This notebook needs a local Ollama server, not an API key -- everything below runs
entirely on your own machine, for free.

**Install Ollama:**

- macOS: `brew install ollama` -- or the installer at [ollama.com/download](https://ollama.com/download)
- Windows: download and run the installer from [ollama.com/download](https://ollama.com/download). It installs Ollama as a background service, so there is no separate `ollama serve` step to run by hand.
- Linux / Ubuntu: `curl -fsSL https://ollama.com/install.sh | sh`

**Start the server** (macOS/Linux only -- the Windows installer already runs this as a service), left running in its own terminal:
```
ollama serve
```

**Pull a model that supports tool calling** -- the same two options Chapter 2's
notebook used (check with `ollama list`): `llama3.1:latest` and `qwen3:8b`.
If you don't have one yet: `ollama pull llama3.1` or `ollama pull qwen3:8b`.

To switch which one the notebook actually calls, edit `OLLAMA_MODEL` in the cell
below -- comment out the active line, uncomment the other.

> **One Ollama-specific gotcha worth knowing before you start:** Ollama's
> OpenAI-compatible endpoint supports `tools` but **not** `tool_choice`. You
> cannot force this local model to call a specific tool the way you can with
> real OpenAI -- which is exactly why every schema in this notebook has to earn
> its call on the strength of its name and description alone. That constraint
> makes Ollama an unusually honest place to practise this chapter.

In [4]:
%pip install openai


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from pathlib import Path

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Models pulled locally (see `ollama list`). Swap the active model by moving
# which line is commented out -- everything else in this notebook stays the same.
OLLAMA_MODEL_LLAMA = "llama3.1:latest"
OLLAMA_MODEL_QWEN3 = "qwen3:8b"

# OLLAMA_MODEL = OLLAMA_MODEL_LLAMA
OLLAMA_MODEL = OLLAMA_MODEL_QWEN3

# We use the standard OpenAI SDK, but point it to the local Ollama server.
# The same code works against real OpenAI by swapping base_url and api_key.
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


def test_connection(client, model_name):
    print(f"Testing connection to local Ollama (model={model_name})...")
    try:
        response = client.chat.completions.create(
            model=model_name, max_tokens=500,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = response.choices[0].message.content
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The assert cells for Exercises 1-6 are pure Python -- they pass without")
        print("Ollama running. The 'Live Demo' cell after each exercise, plus the final")
        print("two sections, all need a live server.")
        print(f"Make sure 'ollama serve' is running and you have pulled: ollama pull {model_name}")


test_connection(client, OLLAMA_MODEL)

Testing connection to local Ollama (model=qwen3:8b)...
  PASS


## The Live-Demo Harness

Every exercise below is graded by an `assert` cell that runs **entirely
offline** -- no Ollama needed, fully deterministic, so a flaky local model can
never mark a correct answer wrong.

After each of those assert cells there is now a **Live Demo** cell that takes
the thing you just wrote and puts it in front of the real model, so you can
watch what that specific fix actually buys you. Those cells need
`ollama serve` running. They are *not* graded and they are allowed to be
messy -- a local 8B model sometimes skips tools or picks the wrong one, which
is itself part of what this chapter is about.

Two small helpers below do the plumbing for all four demos.


In [6]:
def exercise_ready(label, **named_values):
    """
    Guard for demos whose exercise leaves a value as None (Exercise 1).

    Returns True when every named value is filled in. Otherwise names the
    still-empty stub and returns False, so an unattempted exercise prints a
    readable message instead of a confusing traceback.
    """
    missing = [name for name, value in named_values.items() if value is None]
    if not missing:
        return True
    print("-" * 60)
    print(f"SKIPPED -- {label} not done yet")
    print("-" * 60)
    for name in missing:
        print(f"  {name} is still None.")
    print("  Fill in the TODO(s) in the exercise cell above, then re-run this cell.")
    return False


def stub_filled(label, probe):
    """
    Guard for demos whose exercise is an unimplemented function (Exercises 2-4).

    Those stubs raise NotImplementedError until you write them, so we probe
    once with throwaway arguments before spending a real model call.
    """
    try:
        probe()
    except NotImplementedError as exc:
        print("-" * 60)
        print(f"SKIPPED -- {label} not done yet")
        print("-" * 60)
        print(f"  {exc}")
        print("  Implement it in the exercise cell above, then re-run this cell.")
        return False
    except Exception:
        # Any OTHER exception is the learner's own bug, not an unfilled stub --
        # let the demo run so they see it in context.
        pass
    return True


def demo_loop(schemas, dispatch, prompt, max_steps=6, label=None):
    """
    A loud agent loop, built for watching rather than measuring.

    Prints every step's tool call and result so the model's reaction to your
    tool design is visible. Contrast with `run_loop` further down the notebook,
    which is deliberately quiet and returns metrics for the bad-vs-good
    experiment -- different job, so the two are kept separate.

    Returns: {"final_text", "steps", "calls"} where calls is a list of
    (tool_name, arguments_dict, result_string) in the order they happened.
    """
    if label:
        print("-" * 60)
        print(label)
        print("-" * 60)

    messages = [{"role": "user", "content": prompt}]
    calls = []

    for step in range(1, max_steps + 1):
        try:
            response = client.chat.completions.create(
                model=OLLAMA_MODEL, tools=schemas, messages=messages,
            )
        except Exception as exc:
            print(f"  request failed -- {type(exc).__name__}: {exc}")
            print("  (is 'ollama serve' running?)")
            return {"final_text": None, "steps": step - 1, "calls": calls}

        msg = response.choices[0].message

        assistant_msg = {"role": "assistant"}
        if msg.content:
            assistant_msg["content"] = msg.content
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": tc.type,
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            print(f"  STEP {step} | no tool call | the model answered directly")
            print(f"    answer: {(msg.content or '').strip()[:300]}")
            return {"final_text": msg.content, "steps": step, "calls": calls}

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            try:
                args = json.loads(tool_call.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            print(f"  STEP {step} | tool={name} | args={args}")

            fn = dispatch.get(name)
            if fn is None:
                result = f"Error: no such tool '{name}'. Available: {list(dispatch)}."
            else:
                try:
                    result = str(fn(**args))
                except Exception as exc:
                    result = f"Error: {type(exc).__name__}: {exc}"

            print(f"           -> {result[:200]}")
            calls.append((name, args, result))
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    print(f"  hit max_steps={max_steps} without a final answer")
    return {"final_text": None, "steps": max_steps, "calls": calls}


print("-" * 60)
print("DEMO HARNESS READY")
print("-" * 60)
print("  exercise_ready() / stub_filled() -- skip cleanly if an exercise isn't done")
print("  demo_loop()                      -- verbose agent loop for the demos")
print()
print("  The assert cells remain the graded check. Demos are for watching.")


------------------------------------------------------------
DEMO HARNESS READY
------------------------------------------------------------
  exercise_ready() / stub_filled() -- skip cleanly if an exercise isn't done
  demo_loop()                      -- verbose agent loop for the demos

  The assert cells remain the graded check. Demos are for watching.


## The Shared Task and Data

Every tool below reads from one small in-memory task store. Twelve tasks is
deliberately tiny -- small enough that you can verify any tool's output by eye,
big enough that the difference between "return everything" and "return what the
model needs next" is already measurable in tokens.

`REFERENCE_DATE` is fixed rather than `date.today()` so that "overdue" means the
same thing on every run. A tool whose behaviour silently changes with the wall
clock is a tool you cannot write a passing test for.

In [3]:
REFERENCE_DATE = "2026-08-01"  # fixed "as of" date, so "overdue" is reproducible

TASK_STORE = [
    {"id": "task_1",  "title": "Fix login bug",           "assignee": "alex",  "status": "open",   "priority": "high",     "due_date": "2026-07-15", "project": "auth"},
    {"id": "task_2",  "title": "Update docs",             "assignee": "priya", "status": "closed", "priority": "low",      "due_date": "2026-06-01", "project": "docs"},
    {"id": "task_3",  "title": "Migrate database",        "assignee": "sam",   "status": "open",   "priority": "critical", "due_date": "2026-08-05", "project": "infra"},
    {"id": "task_4",  "title": "Design new logo",         "assignee": "priya", "status": "closed", "priority": "medium",   "due_date": "2026-05-20", "project": "marketing"},
    {"id": "task_5",  "title": "Refactor auth module",    "assignee": "alex",  "status": "open",   "priority": "high",     "due_date": "2026-07-20", "project": "auth"},
    {"id": "task_6",  "title": "Write onboarding guide",  "assignee": "sam",   "status": "closed", "priority": "low",      "due_date": "2026-06-15", "project": "docs"},
    {"id": "task_7",  "title": "Fix payment bug",         "assignee": "alex",  "status": "open",   "priority": "critical", "due_date": "2026-07-01", "project": "billing"},
    {"id": "task_8",  "title": "Plan Q3 roadmap",         "assignee": "priya", "status": "open",   "priority": "medium",   "due_date": "2026-08-10", "project": "planning"},
    {"id": "task_9",  "title": "Security audit",          "assignee": "sam",   "status": "open",   "priority": "critical", "due_date": "2026-07-25", "project": "infra"},
    {"id": "task_10", "title": "Update dependencies",     "assignee": "alex",  "status": "closed", "priority": "low",      "due_date": "2026-06-10", "project": "infra"},
    {"id": "task_11", "title": "Customer interview notes","assignee": "priya", "status": "closed", "priority": "medium",   "due_date": "2026-05-30", "project": "research"},
    {"id": "task_12", "title": "Ship rate limiter",       "assignee": "sam",   "status": "open",   "priority": "high",     "due_date": "2026-08-20", "project": "infra"},
]

print(f"TASK_STORE: {len(TASK_STORE)} tasks, as of REFERENCE_DATE={REFERENCE_DATE}")
print(f"  open:   {sum(1 for t in TASK_STORE if t['status'] == 'open')}")
print(f"  closed: {sum(1 for t in TASK_STORE if t['status'] == 'closed')}")
print(f"  overdue (open AND due before {REFERENCE_DATE}): "
      f"{sum(1 for t in TASK_STORE if t['status'] == 'open' and t['due_date'] < REFERENCE_DATE)}")

# The one answer the real-model comparison at the end grades against, computed
# here directly with no model involved.
GROUND_TRUTH_ALEX_OPEN_HIGH = [
    t for t in TASK_STORE
    if t["assignee"] == "alex" and t["status"] == "open" and t["priority"] == "high"
]
print(f"\nGround truth -- alex's open high-priority tasks: "
      f"{[t['title'] for t in GROUND_TRUTH_ALEX_OPEN_HIGH]}")

TASK_STORE: 12 tasks, as of REFERENCE_DATE=2026-08-01
  open:   7
  closed: 5
  overdue (open AND due before 2026-08-01): 4

Ground truth -- alex's open high-priority tasks: ['Fix login bug', 'Refactor auth module']


## Where a Tool Schema Lives in Each Format

Chapter 2's notebook contrasted the two wire formats for the *loop* (`tool_calls`
vs `tool_use`, `finish_reason` vs `stop_reason`). This chapter needs the other
half of that contrast: where the schema itself goes, and what it's called.

| Concept | Anthropic | OpenAI / Ollama (this notebook) |
|---|---|---|
| wrapper around each tool | the tool object itself, flat | `{"type": "function", "function": {...}}` |
| the JSON Schema for arguments | `input_schema` | `parameters` |
| the properties block inside it | `{"type": "object", "properties": {...}, "required": [...]}` | **identical** |
| forcing one specific tool | `tool_choice` | `tool_choice` -- *not supported by Ollama* |
| out-of-band flags (e.g. destructive) | `metadata` | top-level key beside `function`, read by your own orchestrator |

The row that matters most for this chapter is the third one: **the part you
actually spend design effort on is byte-for-byte identical in both formats.**
Names, descriptions, flat-vs-nested arguments, enums, pagination parameters --
every principle in the notes transfers unchanged. What differs is only the
envelope it travels in. That is why the notes can teach one set of design rules
and have them apply to both providers, and why a bad schema is bad in exactly
the same way on either one.

```text
[ THE SAME SCHEMA, TWO ENVELOPES ]

  ANTHROPIC                          OPENAI / OLLAMA
  {                                  {
    "name": "search_tasks",            "type": "function",
    "description": "...",              "function": {
    "input_schema": {  <-------+         "name": "search_tasks",
      "type": "object",        |         "description": "...",
      "properties": {...},     |         "parameters": {  <---+
      "required": [...]        |           "type": "object",  |
    }                          |           "properties": {...},   IDENTICAL
  }                            |           "required": [...]  |   BLOCK
                               +---------- }                  |
                                         }                <---+
                                       }
```

## The Bad Tool Set

Here are three tools that break the rules from Sections 2, 3, and 4 all at once.

The important thing to notice: **all three of them work fine.** Run them and
nothing crashes. No exception, no wrong data, no bug a test would catch. If you
reviewed this code as a normal Python PR you would approve it.

So where does it go wrong? Entirely inside the model's head, when it tries to
figure out *which* tool to call and *what* to put in the arguments. That's the
whole reason this kind of bug is so easy to ship: your tests are green, your
tools return valid JSON, and the agent still behaves badly.

Read each schema below and try to spot the problem yourself before reading the
`# VIOLATION` comment next to it.

### 1. `do_thing` -- nobody can tell what this does

Look at the name: `do_thing`. Do *what* thing? Now the description: `"Does the
thing."` That adds literally nothing. Remember from Section 2 that the name and
description are the *only* information the model gets -- so right now it has
nothing to go on.

Then look at the arguments:

* `x` -- `x` of what? A task id? A person's name? A search word? The model has
  to guess.
* `opts` -- to filter by person, the model has to build a nested structure:
  `opts.filters.meta.who`. That's an object, inside an object, inside an object.
  Nothing in the schema tells the model that `meta`, `who`, or `state` even
  exist, so it has to invent the whole inner shape correctly on the first try.
  This is the "JSON inside JSON" problem from Section 2 -- and every extra layer
  of nesting is another chance to get it wrong.

### 2. `dump_tasks` -- returns everything, always

This tool takes **no arguments at all**. There is no way to ask it for "just
alex's open tasks." It hands back every field of every record, every time.

So the filtering still has to happen -- it just happens in the *worst possible
place*. Instead of one line of Python on the server, the model has to read all
12 full records inside its context window and mentally throw away the ones it
doesn't need. You pay tokens for all that data, on every turn after it arrives
(Section 4), and you're relying on the model to filter correctly by hand.

### 3. `t_upd` -- a secret you never told the model

The name `t_upd` is an abbreviation only its author understands. The description,
`"Updates t."`, is no better.

But the real problem is the `s` argument. Looking at the code, `s` is the new
status, and only `"open"` and `"closed"` are valid values. **Nowhere in the
schema does it say that.** The schema just says `s` is a string.

So the model has to guess the valid values out of thin air. It might send
`"complete"`, `"done"`, or `"Closed"` -- all perfectly reasonable guesses, all
wrong. This is the enum-that-only-exists-in-your-head problem: you know the
rule, the schema doesn't state it, and the model gets blamed for breaking a rule
it was never shown.

---

**The pattern across all three:** every one of these failures is a
*communication* failure, not a code failure. That is exactly what the next
section fixes -- without changing what any tool actually *does*.


In [9]:
def _bad_do_thing(x=None, opts=None):
    """Ambiguous name, ambiguous arg 'x', and a needlessly nested opts shape."""
    opts = opts or {}
    meta = opts.get("filters", {}).get("meta", {})
    who, state = meta.get("who"), meta.get("state")
    matches = [
        t for t in TASK_STORE
        if (who is None or t["assignee"] == who) and (state is None or t["status"] == state)
    ]
    return json.dumps(matches)  # full records, every field, no truncation


def _bad_dump_tasks():
    """No arguments at all -- the model cannot filter, so it gets everything."""
    return json.dumps(TASK_STORE)


def _bad_t_upd(i=None, s=None):
    """Cryptic name, cryptic args, and an undocumented enum for 's'."""
    for t in TASK_STORE:
        if t["id"] == i:
            t["status"] = s
            return json.dumps({"ok": True})
    return json.dumps({"ok": False})


BAD_DISPATCH = {"do_thing": _bad_do_thing, "dump_tasks": _bad_dump_tasks, "t_upd": _bad_t_upd}

BAD_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "do_thing",                      # VIOLATION: says nothing
            "description": "Does the thing.",        # VIOLATION: zero information
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {"type": "string"},         # VIOLATION: 'x' of what?
                    "opts": {                        # VIOLATION: nested JSON-in-JSON
                        "type": "object",
                        "properties": {"filters": {"type": "object"}},
                    },
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "dump_tasks",
            "description": "Dumps tasks.",           # VIOLATION: no filters offered
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "t_upd",                         # VIOLATION: cryptic abbreviation
            "description": "Updates t.",             # VIOLATION: undocumented enum for 's'
            "parameters": {
                "type": "object",
                "properties": {"i": {"type": "string"}, "s": {"type": "string"}},
                "required": ["i", "s"],
            },
        },
    },
]

print("BAD tool set defined:", [s["function"]["name"] for s in BAD_TOOL_SCHEMAS])
print()
print("Note that all three RUN correctly -- here is dump_tasks() returning valid data:")
_preview = _bad_dump_tasks()
print(f"  {len(_preview)} characters of JSON, every field of all {len(TASK_STORE)} records")
print(f"  first 150 chars: {_preview[:150]}...")

BAD tool set defined: ['do_thing', 'dump_tasks', 't_upd']

Note that all three RUN correctly -- here is dump_tasks() returning valid data:
  1857 characters of JSON, every field of all 12 records
  first 150 chars: [{"id": "task_1", "title": "Fix login bug", "assignee": "alex", "status": "open", "priority": "high", "due_date": "2026-07-15", "project": "auth"}, {"...


## The Good Tool Set

The same capabilities, redesigned against Sections 2 through 4. Note what did
*not* change: the underlying data, the Python that filters it, the fact that
there are still roughly three operations. Anthropic's SWE-bench result quoted in
the notes came from exactly this kind of edit -- the tools did not get more
powerful, the interface got clearer.

Four specific fixes, one per principle:

- **Section 2 (naming):** `search_tasks`, `get_task_detail`, `set_task_status` --
  verb-first, unambiguous, and each description states the pagination contract
  and the valid enum values the model would otherwise have to guess.
- **Section 3 (granularity):** the three filters that a human thinks of as one
  question ("alex's open high-priority work") are one call, not three -- but
  `set_task_status` stays a **separate** tool, because it mutates, and Section 3
  is explicit that dangerous operations do not get merged behind a `mode` flag.
- **Section 4 (return values):** a `response_format` enum with `concise` and
  `detailed`, so the caller chooses verbosity per call rather than always paying
  for full records.
- **Section 4 (truncation):** results are paginated with an explicit
  `next_cursor` and a sentence saying how many matches remain -- never a silent
  cliff.

In [6]:
PAGE_SIZE = 5


def _good_search_tasks(assignee=None, status=None, priority=None, cursor=0,
                       response_format="concise"):
    """Consolidated, flat, paginated, verbosity-aware search over TASK_STORE."""
    def status_matches(task):
        if status is None:
            return True
        if status == "overdue":  # a filter the model would otherwise compute by hand
            return task["status"] == "open" and task["due_date"] < REFERENCE_DATE
        return task["status"] == status

    matches = [
        t for t in TASK_STORE
        if (assignee is None or t["assignee"] == assignee)
        and status_matches(t)
        and (priority is None or t["priority"] == priority)
    ]

    page = matches[cursor: cursor + PAGE_SIZE]
    remaining = len(matches) - (cursor + len(page))

    if response_format == "detailed":
        results = page  # every field, only because the caller explicitly asked
    else:
        results = [f"{t['id']}: {t['title']} ({t['priority']}, due {t['due_date']})" for t in page]

    payload = {"results": results, "total_matches": len(matches)}
    if remaining > 0:
        payload["next_cursor"] = cursor + len(page)
        payload["message"] = (
            f"Showing {len(page)} of {len(matches)} matches. {remaining} remain -- "
            f"call again with cursor={cursor + len(page)} to continue."
        )
    else:
        payload["next_cursor"] = None
        payload["message"] = f"All {len(matches)} match(es) shown."
    return json.dumps(payload)


def _good_get_task_detail(task_id, response_format="concise"):
    """Single-record lookup with the same verbosity control."""
    task = next((t for t in TASK_STORE if t["id"] == task_id), None)
    if task is None:
        # Section 5 in miniature: name the problem AND show a valid value.
        return (f"Error: no task with id '{task_id}'. Task ids look like 'task_1' "
                f"through 'task_{len(TASK_STORE)}'. Use search_tasks to find one.")
    if response_format == "detailed":
        return json.dumps(task)
    return f"{task['id']}: {task['title']} ({task['status']}, {task['priority']}, assigned to {task['assignee']})"


def _good_set_task_status(task_id, new_status):
    """
    Idempotent status setter (notes Section 7): calling it twice with the same
    arguments leaves the store in the same state and reports the same result.
    """
    valid = ["open", "closed"]
    if new_status not in valid:
        return f"Error: invalid new_status '{new_status}'. Valid values: {valid}."
    task = next((t for t in TASK_STORE if t["id"] == task_id), None)
    if task is None:
        return f"Error: no task with id '{task_id}'. Use search_tasks to find a valid id."
    already = task["status"] == new_status
    task["status"] = new_status
    return json.dumps({"id": task_id, "status": new_status, "changed": not already})


GOOD_DISPATCH = {
    "search_tasks": _good_search_tasks,
    "get_task_detail": _good_get_task_detail,
    "set_task_status": _good_set_task_status,
}

GOOD_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_tasks",
            "description": (
                "Search the task tracker by any combination of assignee, status, and priority. "
                "Returns up to 5 matches per call as concise one-line summaries; use the "
                "'next_cursor' value from a previous response to page further. Pass "
                "status='overdue' to get open tasks whose due date has already passed -- do not "
                "compute overdue yourself. Prefer narrow filters: an unfiltered search returns "
                "only the first page and wastes a call."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "assignee": {"type": "string", "description": "Username, e.g. 'alex', 'priya', 'sam'."},
                    "status": {
                        "type": "string",
                        "enum": ["open", "closed", "overdue"],
                        "description": "'overdue' means open AND past its due date.",
                    },
                    "priority": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"],
                        "description": "Exact priority to filter on.",
                    },
                    "cursor": {
                        "type": "integer",
                        "description": "Offset to resume from. Use the 'next_cursor' from the previous response; omit for the first page.",
                    },
                    "response_format": {
                        "type": "string",
                        "enum": ["concise", "detailed"],
                        "description": "'concise' (default) returns one-line summaries. Use 'detailed' only when you need full records to pass into another tool.",
                    },
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_task_detail",
            "description": (
                "Fetch one task by its exact id (e.g. 'task_7'). Use search_tasks first if you "
                "do not already have the id. Returns a one-line summary by default; pass "
                "response_format='detailed' for the full record including project and due date."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "task_id": {"type": "string", "description": "Exact task id, e.g. 'task_7'."},
                    "response_format": {"type": "string", "enum": ["concise", "detailed"]},
                },
                "required": ["task_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_task_status",
            "description": (
                "Change one task's status. This MUTATES the tracker. Safe to call twice with the "
                "same arguments (idempotent): the second call reports changed=false and alters "
                "nothing. Kept separate from search_tasks on purpose -- reads and writes do not "
                "share a tool."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "task_id": {"type": "string", "description": "Exact task id, e.g. 'task_7'."},
                    "new_status": {"type": "string", "enum": ["open", "closed"]},
                },
                "required": ["task_id", "new_status"],
            },
        },
    },
]

print("GOOD tool set defined:", [s["function"]["name"] for s in GOOD_TOOL_SCHEMAS])
print()
print("Same question, both verbosity modes -- 'alex, open, high':")
_concise = _good_search_tasks(assignee="alex", status="open", priority="high")
_detailed = _good_search_tasks(assignee="alex", status="open", priority="high", response_format="detailed")
print(f"  concise : {len(_concise):4d} chars -> {_concise}")
# print(f"  detailed: {len(_detailed):4d} chars -> {_detailed[:120]}...")
print(f"  detailed: {len(_detailed):4d} chars -> {_detailed}")
print(f"\n  detailed costs {len(_detailed) / len(_concise):.1f}x the characters of concise,")
print("  for data the model only needs if it plans to act on it further.")
print()
print("Idempotency check (notes Section 7) -- calling set_task_status twice:")
print(f"  call 1: {_good_set_task_status('task_8', 'closed')}")
print(f"  call 2: {_good_set_task_status('task_8', 'closed')}   <- changed=false, nothing doubled")
_good_set_task_status("task_8", "open")  # restore so later cells see the original store

GOOD tool set defined: ['search_tasks', 'get_task_detail', 'set_task_status']

Same question, both verbosity modes -- 'alex, open, high':
  concise :  194 chars -> {"results": ["task_1: Fix login bug (high, due 2026-07-15)", "task_5: Refactor auth module (high, due 2026-07-20)"], "total_matches": 2, "next_cursor": null, "message": "All 2 match(es) shown."}
  detailed:  392 chars -> {"results": [{"id": "task_1", "title": "Fix login bug", "assignee": "alex", "status": "open", "priority": "high", "due_date": "2026-07-15", "project": "auth"}, {"id": "task_5", "title": "Refactor auth module", "assignee": "alex", "status": "open", "priority": "high", "due_date": "2026-07-20", "project": "auth"}], "total_matches": 2, "next_cursor": null, "message": "All 2 match(es) shown."}

  detailed costs 2.0x the characters of concise,
  for data the model only needs if it plans to act on it further.

Idempotency check (notes Section 7) -- calling set_task_status twice:
  call 1: {"id": "task_8", "status"

'{"id": "task_8", "status": "open", "changed": true}'

## Exercise 1 -- Rewrite Three Bad Tool Schemas (OpenAI Format)

`BAD_SCHEMAS_TO_FIX` below holds three more schemas, each with a different
Section-2 violation. Write a corrected version of each as `GOOD_SCHEMA_1`,
`GOOD_SCHEMA_2`, `GOOD_SCHEMA_3`.

You are not implementing anything -- this exercise is purely about the
*interface*. Two things to get right, and the checks below enforce both:

1. **The OpenAI envelope.** Every schema must be `{"type": "function",
   "function": {"name": ..., "description": ..., "parameters": {...}}}`. If you
   have been reading the notes' Anthropic examples, the reflex to write
   `input_schema` at the top level is the exact mistake worth making once here
   rather than against a live API later.
2. **The design principles.** Descriptive names, descriptions long enough to say
   something the name doesn't already say (40+ characters), and flat arguments --
   no property whose `type` is itself `"object"`.

In [15]:
BAD_SCHEMAS_TO_FIX = [
    {
        "type": "function",
        "function": {
            "name": "get",
            "description": "Gets stuff",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "string"}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "upd",
            "description": "Updates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "object",
                        "properties": {"payload": {"type": "object"}},
                    }
                },
                "required": ["data"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "srch",
            "description": "search",
            "parameters": {
                "type": "object",
                "properties": {"q": {"type": "string"}},
                "required": ["q"],
            },
        },
    },
]

# rewrite the below tools schema for this maybe
# DEMO1_PROMPT = "Look up the profile for customer cust_42."

# TODO 1: rewrite BAD_SCHEMAS_TO_FIX[0] ("get") in full OpenAI format -- give it
# a descriptive name (not "get") and a description of at least 40 characters
# that says what it returns, not just that it gets something.
GOOD_SCHEMA_1 = {
    "type": "function",
    "function": {
        "name": "get_customer_profile",
        "description": (
            "Fetch one customer's profile by their internal customer id (e.g. 'cust_42'). "
            "Returns the customer's name, email, signup date, and current plan. Does not "
            "include orders or support tickets -- use get_customer_context for those."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "Internal customer id, e.g. 'cust_42'. Not the email address.",
                }
            },
            "required": ["customer_id"],
        },
    },
}

# TODO 2: rewrite BAD_SCHEMAS_TO_FIX[1] ("upd") -- flatten the nested
# data.payload argument into top-level properties (no property may have
# type "object"), give it a real name and a 40+ character description.
GOOD_SCHEMA_2 = {
    "type": "function",
    "function": {
        "name": "update_customer_contact",
        "description": (
            "Update a customer's contact details. Only the fields you pass are changed; "
            "omitted fields keep their current values. Returns the updated profile. "
            "This mutates the record -- there is no undo."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Internal customer id, e.g. 'cust_42'."},
                "email": {"type": "string", "description": "New email address, e.g. 'alice@example.com'."},
                "phone": {"type": "string", "description": "New phone number in E.164 form, e.g. '+14155550123'."},
                "billing_address": {"type": "string", "description": "Full billing address on one line."},
            },
            "required": ["customer_id"],
        },
    },
}

# TODO 3: rewrite BAD_SCHEMAS_TO_FIX[2] ("srch") -- rename the cryptic 'q'
# parameter to something descriptive, and write a description (40+ chars)
# that mentions what a caller gets back, including any pagination contract.
GOOD_SCHEMA_3 = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": (
            "Search help-centre articles by keyword. Returns up to 10 matching article "
            "titles with their ids and a one-line excerpt each; pass the 'next_cursor' "
            "from a previous response to page further. Prefer specific multi-word queries: "
            "a single common word returns the first page of hundreds of matches."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "search_query": {
                    "type": "string",
                    "description": "Keywords to search for, e.g. 'reset password mobile app'.",
                },
                "cursor": {
                    "type": "integer",
                    "description": "Offset to resume from; use 'next_cursor' from the previous response. Omit for the first page.",
                },
            },
            "required": ["search_query"],
        },
    },
}

In [16]:
def _openai_shaped(schema):
    """True if the schema uses the OpenAI envelope rather than Anthropic's."""
    return (
        schema.get("type") == "function"
        and isinstance(schema.get("function"), dict)
        and "parameters" in schema["function"]
        and "input_schema" not in schema          # the Anthropic reflex
        and "input_schema" not in schema["function"]
    )


def _is_flat(schema):
    """True if no property in function.parameters.properties is itself type 'object'."""
    props = schema["function"]["parameters"].get("properties", {})
    return all(p.get("type") != "object" for p in props.values())


for _n, _s in [(1, GOOD_SCHEMA_1), (2, GOOD_SCHEMA_2), (3, GOOD_SCHEMA_3)]:
    assert _s is not None, f"GOOD_SCHEMA_{_n} is still None -- fill in TODO {_n}"
    assert _openai_shaped(_s), (
        f"GOOD_SCHEMA_{_n} is not in OpenAI format -- it needs "
        f'{{"type": "function", "function": {{..., "parameters": {{...}}}}}} '
        f"and must NOT use Anthropic's 'input_schema'."
    )
    _fn = _s["function"]
    assert len(_fn["description"]) >= 40, (
        f"GOOD_SCHEMA_{_n}'s description is {len(_fn['description'])} chars -- too short to "
        f"say anything the name doesn't already say (aim for 40+)."
    )

assert GOOD_SCHEMA_1["function"]["name"] != "get", "GOOD_SCHEMA_1 still has the ambiguous name 'get'"
assert GOOD_SCHEMA_2["function"]["name"] != "upd", "GOOD_SCHEMA_2 still has the ambiguous name 'upd'"
assert GOOD_SCHEMA_3["function"]["name"] != "srch", "GOOD_SCHEMA_3 still has the ambiguous name 'srch'"

assert _is_flat(GOOD_SCHEMA_2), (
    "GOOD_SCHEMA_2 still nests an object property -- flatten data.payload into "
    "explicit top-level parameters (notes Section 2, 'Argument Shape')."
)

_q_props = GOOD_SCHEMA_3["function"]["parameters"]["properties"]
assert "q" not in _q_props, "GOOD_SCHEMA_3 still uses the cryptic parameter name 'q'"

print("-" * 60)
print("PASS -- all three rewrites are OpenAI-shaped, named, described, and flat")
print("-" * 60)
for _n, _s in [(1, GOOD_SCHEMA_1), (2, GOOD_SCHEMA_2), (3, GOOD_SCHEMA_3)]:
    _fn = _s["function"]
    _before = BAD_SCHEMAS_TO_FIX[_n - 1]["function"]
    print(f"  {_n}. {_before['name']!r} -> {_fn['name']!r}")
    print(f"     description: {len(_before['description'])} chars -> {len(_fn['description'])} chars")
    print(f"     parameters:  {list(_before['parameters'].get('properties', {}))} -> {list(_fn['parameters'].get('properties', {}))}")

------------------------------------------------------------
PASS -- all three rewrites are OpenAI-shaped, named, described, and flat
------------------------------------------------------------
  1. 'get' -> 'get_customer_profile'
     description: 10 chars -> 225 chars
     parameters:  ['id'] -> ['customer_id']
  2. 'upd' -> 'update_customer_contact'
     description: 8 chars -> 190 chars
     parameters:  ['data'] -> ['customer_id', 'email', 'phone', 'billing_address']
  3. 'srch' -> 'search_knowledge_base'
     description: 6 chars -> 293 chars
     parameters:  ['q'] -> ['search_query', 'cursor']


### Live Demo -- Watch the Model Choose Between Your Bad and Good Schemas

*Not graded; needs `ollama serve`.* The assert above proved your rewrites are
well-formed. This shows what that well-formedness is **for**: the same request,
offered the vague `get` / `upd` / `srch` schemas and then your rewritten ones,
with nothing changed but the names and descriptions.

Section 2's claim was that the name, the parameter names, and the description
are the entire specification the model ever sees. Here that stops being a
claim.


In [18]:
DEMO1_PROMPT = (
    "I don't have an exact record handy, but I think the customer is cust_42 -- "
    "can you look that up for me?"
)

def show_tool_choice(label, schemas):
    print("-" * 60)
    print(f"TOOL-CHOICE DEMO: {label}")
    print("-" * 60)
    try:
        response = client.chat.completions.create(
            model=OLLAMA_MODEL, tools=schemas,
            messages=[{"role": "user", "content": DEMO1_PROMPT}],
        )
    except Exception as exc:
        print(f"  request failed -- {type(exc).__name__}: {exc}")
        print("  (is 'ollama serve' running?)")
        return

    msg = response.choices[0].message
    if not msg.tool_calls:
        print("  NOTE: the real model didn't call any tools this run.")
        print(f"    it said: {(msg.content or '').strip()[:200]}")
        return

    for tool_call in msg.tool_calls:
        try:
            args = json.loads(tool_call.function.arguments or "{}")
        except json.JSONDecodeError:
            args = tool_call.function.arguments
        print(f"  chose tool: {tool_call.function.name}")
        print(f"  arguments : {args}")

        # NEW: does the ID argument actually come out clean?
        id_value = args.get("id") if isinstance(args, dict) else None
        if id_value is None:
            for key in ("customer_id", "id_value"):
                if isinstance(args, dict) and key in args:
                    id_value = args[key]
        if id_value is not None:
            clean = id_value == "cust_42"
            print(f"  id argument exactly 'cust_42'? {clean}  (got: {id_value!r})")


if exercise_ready(
    "Exercise 1",
    GOOD_SCHEMA_1=GOOD_SCHEMA_1, GOOD_SCHEMA_2=GOOD_SCHEMA_2, GOOD_SCHEMA_3=GOOD_SCHEMA_3,
):
    print(f"Question asked both times: {DEMO1_PROMPT!r}")
    print()
    show_tool_choice("the BAD schemas (get / upd / srch)", BAD_SCHEMAS_TO_FIX)
    print()
    show_tool_choice("YOUR rewritten schemas", [GOOD_SCHEMA_1, GOOD_SCHEMA_2, GOOD_SCHEMA_3])
    print()
    print("How to read this:")
    print("  * With the bad set, 'get' is the only plausible match but says nothing")
    print("    about customers -- so the model either picks it for lack of an")
    print("    alternative, picks 'srch' instead, or invents an argument name.")
    print("  * With your set, the name and description say what comes back, so the")
    print("    choice is forced rather than guessed.")
    print("  * If BOTH runs pick correctly: that's a real result, not a failed demo.")
    print("    Three tools is a small enough field that an 8B model can often")
    print("    brute-force it -- Section 9 is about what happens at 20+ tools, which")
    print("    is exactly what the bad-vs-good experiment further down measures.")


Question asked both times: "I don't have an exact record handy, but I think the customer is cust_42 -- can you look that up for me?"

------------------------------------------------------------
TOOL-CHOICE DEMO: the BAD schemas (get / upd / srch)
------------------------------------------------------------
  chose tool: get
  arguments : {'id': 'cust_42'}
  id argument exactly 'cust_42'? True  (got: 'cust_42')

------------------------------------------------------------
TOOL-CHOICE DEMO: YOUR rewritten schemas
------------------------------------------------------------
  chose tool: get_customer_profile
  arguments : {'customer_id': 'cust_42'}
  id argument exactly 'cust_42'? True  (got: 'cust_42')

How to read this:
  * With the bad set, 'get' is the only plausible match but says nothing
    about customers -- so the model either picks it for lack of an
    alternative, picks 'srch' instead, or invents an argument name.
  * With your set, the name and description say what comes bac

>> NOTE: since the above query was to ask for cutomer details via id it has only one choice so both good and bad used the same tool. Below we have a more overlapping bad chema that will be more demoable

In [ ]:
DEMO1C_PROMPT = "Look up the profile for customer cust_42."

def _fake_profile(id): return json.dumps({"id": id, "name": "Dana Kim", "plan": "pro"})
def _fake_orders(id): return json.dumps({"id": id, "orders": ["ord_1001", "ord_1002"]})
def _fake_billing(id): return json.dumps({"id": id, "balance_due": 42.50})

OVERLAPPING_BAD_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_customer", "description": "Gets customer information.",
        "parameters": {"type": "object", "properties": {"id": {"type": "string"}}, "required": ["id"]},
    }},
    {"type": "function", "function": {
        "name": "customer_lookup", "description": "Looks up customer information.",
        "parameters": {"type": "object", "properties": {"id": {"type": "string"}}, "required": ["id"]},
    }},
    {"type": "function", "function": {
        "name": "fetch_customer", "description": "Fetches customer information.",
        "parameters": {"type": "object", "properties": {"id": {"type": "string"}}, "required": ["id"]},
    }},
]
OVERLAPPING_BAD_DISPATCH = {
    "get_customer": _fake_profile,        # this one actually IS the profile
    "customer_lookup": _fake_orders,      # this one returns order history
    "fetch_customer": _fake_billing,      # this one returns billing info
}

OVERLAPPING_GOOD_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_customer_profile", "description": "Returns name, email, and plan for a customer, given their customer_id.",
        "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]},
    }},
    {"type": "function", "function": {
        "name": "get_customer_orders", "description": "Returns the order-id list for a customer, given their customer_id. Does not include profile fields.",
        "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]},
    }},
    {"type": "function", "function": {
        "name": "get_customer_billing", "description": "Returns the outstanding balance for a customer, given their customer_id. Does not include profile fields.",
        "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]},
    }},
]

def show_tool_choice_and_payload(label, schemas, dispatch=None):
    print("-" * 60); print(f"TOOL-CHOICE DEMO: {label}"); print("-" * 60)
    response = client.chat.completions.create(
        model=OLLAMA_MODEL, tools=schemas,
        messages=[{"role": "user", "content": DEMO1C_PROMPT}],
    )
    msg = response.choices[0].message
    if not msg.tool_calls:
        print(f"  no tool called; it said: {(msg.content or '').strip()[:200]}"); return
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        print(f"  chose tool: {tc.function.name}   arguments: {args}")
        if dispatch and tc.function.name in dispatch:
            print(f"  data it actually got back: {dispatch[tc.function.name](**args)}")

for _ in range(3):
    show_tool_choice_and_payload("OVERLAPPING bad set (3 tools, all say 'customer information')",
                                  OVERLAPPING_BAD_SCHEMAS, OVERLAPPING_BAD_DISPATCH)
show_tool_choice_and_payload("OVERLAPPING good set (each says which fields it returns)",
                              OVERLAPPING_GOOD_SCHEMAS)

------------------------------------------------------------
TOOL-CHOICE DEMO: OVERLAPPING bad set (3 tools, all say 'customer information')
------------------------------------------------------------
  chose tool: customer_lookup   arguments: {'id': 'cust_42'}
  data it actually got back: {"id": "cust_42", "orders": ["ord_1001", "ord_1002"]}
------------------------------------------------------------
TOOL-CHOICE DEMO: OVERLAPPING bad set (3 tools, all say 'customer information')
------------------------------------------------------------
  chose tool: customer_lookup   arguments: {'id': 'cust_42'}
  data it actually got back: {"id": "cust_42", "orders": ["ord_1001", "ord_1002"]}
------------------------------------------------------------
TOOL-CHOICE DEMO: OVERLAPPING bad set (3 tools, all say 'customer information')
------------------------------------------------------------
  chose tool: customer_lookup   arguments: {'id': 'cust_42'}
  data it actually got back: {"id": "cust_42"

## Exercise 2 -- Truncate With a Continuation Handle, Not a Silent Cliff

`LOG_LINES` below is a small log with 40 entries, 25 of which contain `"ERROR"`.
Implement `search_logs_paginated(query, cursor=0, page_size=10)` following notes
Section 4's rule: **never silently truncate.** A model that receives 10 of 25
matches with no indication the other 15 exist will confidently answer as though
it saw everything -- and it will be wrong, through no fault of its own.

Return a dict with:

* `"results"`: up to `page_size` matching lines, starting at `cursor`
* `"next_cursor"`: the offset to continue from, or `None` if nothing remains
* `"message"`: when `next_cursor` is not `None`, a sentence stating how many
  matches remain **and** how to get them; when it is `None`, a sentence stating
  the total that was shown

In [24]:
LOG_LINES = [f"{i:03d} INFO  service started ok" for i in range(15)]
LOG_LINES += [f"{i:03d} ERROR connection timeout on attempt {i}" for i in range(15, 40)]
LOG_LINES.sort()  # interleave INFO/ERROR lines like a real log would be


def search_logs_paginated(query, cursor=0, page_size=10):
    """
    Search LOG_LINES for lines containing `query`, paginated.

    Returns: {"results": [...], "next_cursor": int|None, "message": str}
    Math note: with M total matches and page size P, there are ceil(M/P) pages;
    this call returns the page starting at `cursor` and reports how many of the
    M matches remain after it.
    """
    # TODO 4: find every line in LOG_LINES containing `query` (a case-sensitive
    # substring match is fine), slice out the page starting at `cursor` of length
    # `page_size`, and build the result dict described above. The "message" must
    # name the number remaining and tell the caller which cursor to use next.
    matches = [line for line in LOG_LINES if query in line]
    page = matches[cursor: cursor + page_size]
    shown_through = cursor + len(page)
    remaining = len(matches) - shown_through

    if remaining > 0:
        return {
            "results" : page,
            "next_cursor" : shown_through,
            "message": (
                f"Showing matches {cursor + 1}-{shown_through} of {len(matches)}. "
                f"{remaining} more match(es) remain -- call again with "
                f"cursor={shown_through} to continue."
            )
        }
    return {
        "results": page,
        "next_cursor": None,
        "message": f"All {len(matches)} match(es) shown. No more results.",
    }

In [25]:
page_1 = search_logs_paginated("ERROR", cursor=0, page_size=10)
assert len(page_1["results"]) == 10, f"expected 10 results on page 1, got {len(page_1['results'])}"
assert page_1["next_cursor"] == 10, f"expected next_cursor=10, got {page_1['next_cursor']}"
assert "15" in page_1["message"], "page 1's message should mention the 15 remaining matches"
assert "10" in page_1["message"], "page 1's message should tell the caller which cursor to use next"

page_2 = search_logs_paginated("ERROR", cursor=page_1["next_cursor"], page_size=10)
assert len(page_2["results"]) == 10, f"expected 10 results on page 2, got {len(page_2['results'])}"
assert page_2["next_cursor"] == 20, f"expected next_cursor=20, got {page_2['next_cursor']}"

page_3 = search_logs_paginated("ERROR", cursor=page_2["next_cursor"], page_size=10)
assert len(page_3["results"]) == 5, f"expected 5 results on the final page, got {len(page_3['results'])}"
assert page_3["next_cursor"] is None, "next_cursor must be None once nothing remains"
assert "25" in page_3["message"], "the final message should state the total of 25 matches"

no_match = search_logs_paginated("KERNEL_PANIC")
assert no_match["results"] == [], "a query matching nothing should return an empty list"
assert no_match["next_cursor"] is None, "a query matching nothing has no next page"

print("-" * 60)
print("PASS -- pagination is honest at every boundary, including the last page")
print("-" * 60)
for _label, _page in [("page 1", page_1), ("page 2", page_2), ("page 3", page_3)]:
    print(f"  {_label}: {len(_page['results'])} results, next_cursor={_page['next_cursor']}")
    print(f"          message -> {_page['message']}")
print()
print("The message field is the whole point: a model reading page 1 knows,")
print("without guessing, that it has seen 10 of 25 and exactly how to get the rest.")

------------------------------------------------------------
PASS -- pagination is honest at every boundary, including the last page
------------------------------------------------------------
  page 1: 10 results, next_cursor=10
          message -> Showing matches 1-10 of 25. 15 more match(es) remain -- call again with cursor=10 to continue.
  page 2: 10 results, next_cursor=20
          message -> Showing matches 11-20 of 25. 5 more match(es) remain -- call again with cursor=20 to continue.
  page 3: 5 results, next_cursor=None
          message -> All 25 match(es) shown. No more results.

The message field is the whole point: a model reading page 1 knows,
without guessing, that it has seen 10 of 25 and exactly how to get the rest.


In [33]:
pretty_log_lines = json.dumps(LOG_LINES, indent=2)
print(f"pretty print log lines:\n{pretty_log_lines}")

pretty print log lines:
[
  "000 INFO  service started ok",
  "001 INFO  service started ok",
  "002 INFO  service started ok",
  "003 INFO  service started ok",
  "004 INFO  service started ok",
  "005 INFO  service started ok",
  "006 INFO  service started ok",
  "007 INFO  service started ok",
  "008 INFO  service started ok",
  "009 INFO  service started ok",
  "010 INFO  service started ok",
  "011 INFO  service started ok",
  "012 INFO  service started ok",
  "013 INFO  service started ok",
  "014 INFO  service started ok",
  "015 ERROR connection timeout on attempt 15",
  "016 ERROR connection timeout on attempt 16",
  "017 ERROR connection timeout on attempt 17",
  "018 ERROR connection timeout on attempt 18",
  "019 ERROR connection timeout on attempt 19",
  "020 ERROR connection timeout on attempt 20",
  "021 ERROR connection timeout on attempt 21",
  "022 ERROR connection timeout on attempt 22",
  "023 ERROR connection timeout on attempt 23",
  "024 ERROR connection timeout 

### Live Demo -- Watch the Model Follow Your Continuation Handle

*Not graded; needs `ollama serve`.* Your `search_logs_paginated` returns 10 of
25 matches plus a `message` saying how many remain and which cursor to use.
Nothing *forces* the model to act on that sentence -- so here we find out
whether it does.

Watch the `cursor` argument across steps. If your `message` is doing its job,
it walks 0 -> 10 -> 20 on its own and reports 25. A silent truncation would
have produced a confident, wrong answer of 10 on the first step.


In [ ]:
LOG_SEARCH_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_logs",
            "description": (
                "Search the server log for lines containing a query string. Returns at "
                "most 10 matches per call. When more matches remain, the response's "
                "'message' field states how many are left and which cursor value to "
                "pass to continue from where you stopped."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Substring to search for, e.g. 'ERROR'."},
                    "cursor": {
                        "type": "integer",
                        "description": (
                            "Offset to resume from. Use the 'next_cursor' from the previous "
                            "response; omit it for the first page."
                        ),
                    },
                },
                "required": ["query"],
            },
        },
    },
]


def _demo_search_logs(query, cursor=None, **_):
    """Adapter: your exercise function, page size fixed so only cursor varies."""
    cursor = cursor if cursor is not None else 0
    return json.dumps(search_logs_paginated(query, cursor=cursor, page_size=15))


DEMO2_PROMPT = (
    "How many lines in the log contain 'ERROR'? Page through every result until "
    "none remain, then tell me the exact total. Use the tools -- do not guess."
)

if stub_filled("Exercise 2", lambda: search_logs_paginated("ERROR", cursor=0, page_size=10)):
    print("-" * 60)
    print("CONTINUATION-HANDLE DEMO: search_logs")
    print("-" * 60)
    print(f"PROMPT sent to the model:\n  {DEMO2_PROMPT!r}")

    messages = [{"role": "user", "content": DEMO2_PROMPT}]

    for step in range(1, 7):
        response = client.chat.completions.create(
            model=OLLAMA_MODEL, tools=LOG_SEARCH_SCHEMAS, messages=messages,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"\nSTEP {step}: no more tool calls -- model gave its final answer.")
            print(f"  final answer: {msg.content}")
            break

        messages.append(msg)
        for tool_call in msg.tool_calls:
            args = json.loads(tool_call.function.arguments or "{}")
            print(f"\nSTEP {step}: model called search_logs with arguments: {args}")

            result = _demo_search_logs(**args)
            pretty_result = json.dumps(json.loads(result), indent=2)
            print(f"  tool returned:\n{pretty_result}")

            # we are appending the complete result in the messages see as a tool
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })
    else:
        print("\nstopped after 6 steps without a final answer -- check for a loop.")


------------------------------------------------------------
CONTINUATION-HANDLE DEMO: search_logs
------------------------------------------------------------
PROMPT sent to the model:
  "How many lines in the log contain 'ERROR'? Page through every result until none remain, then tell me the exact total. Use the tools -- do not guess."

STEP 1: model called search_logs with arguments: {'query': 'ERROR', 'cursor': 0}
  tool returned:
{
  "results": [
    "015 ERROR connection timeout on attempt 15",
    "016 ERROR connection timeout on attempt 16",
    "017 ERROR connection timeout on attempt 17",
    "018 ERROR connection timeout on attempt 18",
    "019 ERROR connection timeout on attempt 19",
    "020 ERROR connection timeout on attempt 20",
    "021 ERROR connection timeout on attempt 21",
    "022 ERROR connection timeout on attempt 22",
    "023 ERROR connection timeout on attempt 23",
    "024 ERROR connection timeout on attempt 24",
    "025 ERROR connection timeout on attempt 

> the cursor numbering is a bit weird - this is just for demo - i guess in a real application we will use some kind of DB anyways. Look at the above code block where we are printing the log lines and the confusion of the cursor starting 15 again for step 2 will be cleared - its only taking into account the error log lines

### What This Demo Is Actually Showing

The one-sentence version: **the tool can't return 25 results in one shot, so it hands back a hint telling the model how to come back for more — and this demo proves the model actually reads that hint instead of just believing it already has everything.**

#### The problem this solves

If `search_logs` just dumped all 25 matching lines in one response, that's fine for 25 lines — but real logs have millions. No tool can safely return "everything" in one call. So instead, the tool returns a **page** (here, up to 15 results) plus a `message` telling the model two things: how many are left, and what `cursor` value to send next to keep going. This is the same idea as a "Load More" button on a website — the page can't show you everything at once, so it hands you a token that means "click here to continue from where you stopped."

The risk this demo is checking for: nothing stops the model from reading page 1, seeing "10 more remain," and just answering "15" anyway because it got impatient or didn't parse the `message` field. That would be a **confident wrong answer** — the tool did its job correctly, but the model ignored the instructions inside the data.

#### The flow, as a diagram

```text
 you                     model                      your tool
  |                        |                              |
  |-- "how many ERROR" --->|                              |
  |                        |-- search_logs(cursor=0) ---->|
  |                        |<-- 15 results +              |
  |                        |    next_cursor=15,           |
  |                        |    "10 more remain" ---------|
  |                        |                              |
  |                  [reads message,                      |
  |                   decides to continue]                |
  |                        |                              |
  |                        |-- search_logs(cursor=15) --->|
  |                        |<-- 10 results +              |
  |                        |    "no more remain" ---------|
  |                        |                              |
  |                  [stops calling tools,                |
  |                   counts everything it saw]           |
  |                        |                              |
  |<-- "25 lines total" ---|                              |
```

Notice the model is the one deciding when to stop — nothing in the code forces a second call. The only reason it makes one is that the message field told it there was more to find.

#### Walking through the actual numbers
With page_size=15 and 25 total matching lines, here's exactly what each round trip carries:

```bash
Call 1:  cursor sent = 0
         tool returns: 15 results, next_cursor = 15
         message = "Showing matches 1-15 of 25. 10 more match(es)
                     remain -- call again with cursor=15 to continue."

Call 2:  cursor sent = 15
         tool returns: 10 results, next_cursor = none
         message = "Showing matches 16-25 of 25. 0 more match(es) remain."

Call 3:  no tool call -- model answers in plain text: "25"
```

The cursor value on Call 2 isn't something the model invented — it copied the exact number (15) out of the message string it got back from Call 1. That copy-the-number-forward behavior is the continuation handle. If you ever see the model send a cursor that doesn't match the next_cursor it was just given, that's the bug this whole exercise exists to catch.

#### How this maps to the code below

Each STEP n line you see printed corresponds to one full trip around the for step in range(1, 7): loop:

1. client.chat.completions.create(...) sends the model the conversation so far, plus the search_logs tool description.
2. If the model wants data, it comes back with tool_calls instead of a text answer — that's the if not msg.tool_calls check failing, so we go into the for tool_call in msg.tool_calls: branch.
3. We run the real Python function (_demo_search_logs) and get back JSON — the model never runs any code itself, it only ever sees text.
4. That JSON result is appended to messages with "role": "tool", so on the next loop iteration the model can see what it just got back and decide whether to call the tool again.
5. Once the model has enough information, it replies with plain text and no tool_calls — that's when the loop's if not msg.tool_calls: branch triggers and prints the final answer.

#### A common gotcha
You already hit this one: some local models send "cursor": null on the very first call instead of just leaving the argument out. If your adapter function only relies on a Python default (cursor=0), that default gets silently overridden by the explicit None the model sent — which is why _demo_search_logs coalesces None to 0 by hand rather than trusting the default alone.

#### Where this connects next
This same "return a hint, let the caller decide to continue" shape is exactly what MCP's pagination and the stateless-spec request_state sticky note are doing at the protocol level — the model-facing pattern here is a small-scale version of the same idea you'll see again in Section 10.

## Exercise 3 -- Errors as Instructions, Not Postmortems

Implement `validate_tool_input(input_dict, schema)`. It checks whether the
input for a `create_task` call is valid, using the rules in
`CREATE_TASK_SCHEMA`: `priority` has to be one of a fixed set of values, and
`due_date` has to look like `YYYY-MM-DD`. Return `(True, None)` when the input
is fine, or `(False, error_message)` when it isn't.

Here's the part that makes this different from ordinary form validation: the
thing reading `error_message` is a model that gets **exactly one more try**,
not a person who can open a debugger and go looking. So the message itself has
to do all the work of getting the next attempt right.

Compare these two rejections for the same bad input:

```bash
"Error: invalid priority"
"Invalid value for 'priority': got 'urgent'. Valid values: 'low', 'medium', 'high', 'critical'."
```

The first one only tells the model it failed -- on the retry, it's still
guessing. The second one tells the model *what it sent*, *why it was wrong*,
and *what a correct value looks like* -- so the retry is an informed fix, not
another blind stab. Every error your function returns should look like the
second one: name the field, show the value that was actually received, and
show a valid example.

One more thing worth knowing, since it explains why this matters even more on
this format specifically: Anthropic's API has a dedicated `is_error: true`
flag it attaches to a failed tool result, separate from the message text. In
OpenAI/Ollama format there is no such flag -- the error text is the *entire*
signal the model gets that something went wrong. If the wording is vague here,
there's no backup channel carrying the "this failed" information; the model
has to infer it from the words alone.



In [34]:
import re

CREATE_TASK_SCHEMA = {
    "required": ["title", "priority", "due_date"],
    "priority_enum": ["low", "medium", "high", "critical"],
    "due_date_pattern": r"^\d{4}-\d{2}-\d{2}$",
}


def validate_tool_input(input_dict, schema):
    """
    Validate `input_dict` against `schema`.

    Returns (True, None) if valid.
    Returns (False, error_message) if invalid, where error_message names the
    offending field, states what was actually received, and states what a valid
    value looks like (notes Section 5 -- an error is an instruction for the next
    retry, not a diagnostic for a human reading logs tomorrow).
    """
    # TODO 5: check every field in schema["required"] is present in input_dict;
    # if "priority" is present, check it is one of schema["priority_enum"];
    # if "due_date" is present, check it matches schema["due_date_pattern"].
    # Return (False, <actionable message>) on the first problem found, else
    # (True, None). Each message must include BOTH the bad value received AND
    # a concrete example of a correct one.
    
    # checking for missing required field
    for field in schema["required"]:
        if field not in input_dict:
            return False, (
                f"Missing required field '{field}'. A valid call includes all of: "
                f"{schema['required']}. Example: "
                f"{{\"title\": \"Renew SSL cert\", \"priority\": \"high\", "
                f"\"due_date\": \"2026-07-30\"}}"
            )

    # check for correct enum for priority
    priority = input_dict.get("priority")
    if priority is not None and priority not in schema["priority_enum"]:
        return False, (
            f"Invalid value for 'priority': got {priority!r}. "
            f"Valid values: {', '.join(repr(p) for p in schema['priority_enum'])}."
        )

    # check due date correct format
    due_date = input_dict.get("due_date")
    if due_date is not None and not re.match(schema["due_date_pattern"], str(due_date)):
        return False, (
            f"Invalid date format for 'due_date': got {due_date!r}. "
            f"Use ISO 8601 'YYYY-MM-DD' (e.g. '2026-07-30')."
        )

    return True, None
    

In [35]:
bad_input = {"title": "Renew SSL cert", "priority": "urgent", "due_date": "07/30/2026"}
valid, error = validate_tool_input(bad_input, CREATE_TASK_SCHEMA)
assert valid is False, "bad_input should fail validation (bad priority AND bad date)"
assert "priority" in error.lower(), "the error should name the 'priority' field"
assert "urgent" in error, "the error should echo back the value that was actually received"
assert "critical" in error and "high" in error, "the error should list the valid priority values"

good_input = {"title": "Renew SSL cert", "priority": "high", "due_date": "2026-07-30"}
valid, error = validate_tool_input(good_input, CREATE_TASK_SCHEMA)
assert valid is True and error is None, f"good_input should pass, got error={error!r}"

missing = {"priority": "high", "due_date": "2026-07-30"}
valid, error = validate_tool_input(missing, CREATE_TASK_SCHEMA)
assert valid is False and "title" in error.lower(), "a missing 'title' should be caught and named"

bad_date = {"title": "X", "priority": "low", "due_date": "30-07-2026"}
valid, error = validate_tool_input(bad_date, CREATE_TASK_SCHEMA)
assert valid is False, "'30-07-2026' does not match YYYY-MM-DD and should fail"
assert "2026-07-30" in error or "YYYY-MM-DD" in error, "the error should show the expected format"

print("-" * 60)
print("PASS -- every rejection names the field, the bad value, and a valid one")
print("-" * 60)
print()
print("REPAIR-LOOP DEMO -- what a model actually experiences:")
print()
attempt_1 = {"title": "Renew SSL cert", "priority": "urgent", "due_date": "07/30/2026"}
valid, error = validate_tool_input(attempt_1, CREATE_TASK_SCHEMA)
print(f"  attempt 1: {attempt_1}")
print(f"    -> valid={valid}")
print(f"    -> tool message the model reads back: {error}")
print()
print("  (a model reading that has everything it needs -- it knows 'urgent' was")
print("   rejected, it knows the four values that are legal, and on the NEXT")
print("   field it will also learn the date format. Attempt 2:)")
print()
attempt_2 = {"title": "Renew SSL cert", "priority": "high", "due_date": "07/30/2026"}
valid, error = validate_tool_input(attempt_2, CREATE_TASK_SCHEMA)
print(f"  attempt 2: {attempt_2}")
print(f"    -> valid={valid}")
print(f"    -> tool message: {error}")
print()
attempt_3 = {"title": "Renew SSL cert", "priority": "high", "due_date": "2026-07-30"}
valid, error = validate_tool_input(attempt_3, CREATE_TASK_SCHEMA)
print(f"  attempt 3: {attempt_3}")
print(f"    -> valid={valid}, error={error}")
print()
print("  Three turns, zero human intervention. With 'Error: bad request' as the")
print("  message, attempt 2 would have been another blind guess.")

------------------------------------------------------------
PASS -- every rejection names the field, the bad value, and a valid one
------------------------------------------------------------

REPAIR-LOOP DEMO -- what a model actually experiences:

  attempt 1: {'title': 'Renew SSL cert', 'priority': 'urgent', 'due_date': '07/30/2026'}
    -> valid=False
    -> tool message the model reads back: Invalid value for 'priority': got 'urgent'. Valid values: 'low', 'medium', 'high', 'critical'.

  (a model reading that has everything it needs -- it knows 'urgent' was
   rejected, it knows the four values that are legal, and on the NEXT
   field it will also learn the date format. Attempt 2:)

  attempt 2: {'title': 'Renew SSL cert', 'priority': 'high', 'due_date': '07/30/2026'}
    -> valid=False
    -> tool message: Invalid date format for 'due_date': got '07/30/2026'. Use ISO 8601 'YYYY-MM-DD' (e.g. '2026-07-30').

  attempt 3: {'title': 'Renew SSL cert', 'priority': 'high', 'due_date': 

### Live Demo -- Watch the Model Repair Itself From Your Error Message

*Not graded; needs `ollama serve`.* This is the sharpest demo in the notebook,
because it runs the same broken request twice against two tools that differ
**only in what their error message says**.

Note what the `create_task` schema below deliberately leaves out: no `enum` on
`priority`, no format hint on `due_date`. The model has no way to get it right
up front -- the only path to a valid call is reading the error it gets back.
One version returns your instructional error; the other returns
`"Error: 400 Bad Request"`. Same model, same prompt, same validator underneath.


In [36]:
# Deliberately under-specified: no enum on priority, no format hint on due_date.
# The ONLY way the model can learn the rules is from the error it gets back.
CREATE_TASK_TOOL = [
    {
        "type": "function",
        "function": {
            "name": "create_task",
            "description": "Create a new task in the tracker. Requires a title, a priority, and a due date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "priority": {"type": "string"},
                    "due_date": {"type": "string"},
                },
                "required": ["title", "priority", "due_date"],
            },
        },
    },
]


def _create_task_instructive(**kwargs):
    """Rejects with YOUR error message -- the one that says how to fix it."""
    valid, error = validate_tool_input(kwargs, CREATE_TASK_SCHEMA)
    if not valid:
        return f"Error: {error}"
    return json.dumps({"status": "created", **kwargs})


def _create_task_blunt(**kwargs):
    """Rejects with the error a normal REST API would return. Same validator."""
    valid, _error = validate_tool_input(kwargs, CREATE_TASK_SCHEMA)
    if not valid:
        return "Error: 400 Bad Request"
    return json.dumps({"status": "created", **kwargs})

def run_repair_loop(schemas, dispatch, prompt, max_steps=6):
    messages = [{"role": "user", "content": prompt}]
    calls = []
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=OLLAMA_MODEL, tools=schemas, messages=messages)
        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"  STEP {step}: no more tool calls -- final answer: {msg.content}")
            break

        messages.append(msg)
        for tool_call in msg.tool_calls:
            args = json.loads(tool_call.function.arguments or "{}")
            result = dispatch[tool_call.function.name](**args)
            calls.append((tool_call.function.name, args, result))
            print(f"  STEP {step}: called {tool_call.function.name} with {args}")
            print(f"    -> {result}")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    return calls


DEMO3_PROMPT = (
    "Create a task titled 'Renew SSL cert'. It is urgent, and it is due on "
    "July 30th 2026. Use the create_task tool."
)

if stub_filled("Exercise 3", lambda: validate_tool_input({}, CREATE_TASK_SCHEMA)):
    print(f"Request (phrased to induce an invalid priority AND a US-format date):")
    print(f"  {DEMO3_PROMPT}")

    results = {}
    for label, fn in [("instructive", _create_task_instructive), ("blunt", _create_task_blunt)]:
        print()
        print("-" * 60)
        print(f"REPAIR-LOOP DEMO: create_task ({label} errors)")
        print("-" * 60)
        results[label] = run_repair_loop(CREATE_TASK_TOOL, {"create_task": fn}, DEMO3_PROMPT)

    print()
    print("-" * 60)
    print("SUMMARY")
    print("-" * 60)
    for label, calls in results.items():
        succeeded = any("created" in result for _n, _a, result in calls)
        print(f"  {label:12s} attempts={len(calls)} succeeded={succeeded}")
    print()
    print("How to read this:")
    print("  * The instructive run should reach a valid call in 2-3 attempts.")
    print("  * The blunt run has nothing to learn from -- it re-guesses blind.")
    print("  * Attempt COUNT is the metric, not success -- same validator both runs.")



Request (phrased to induce an invalid priority AND a US-format date):
  Create a task titled 'Renew SSL cert'. It is urgent, and it is due on July 30th 2026. Use the create_task tool.

------------------------------------------------------------
REPAIR-LOOP DEMO: create_task (instructive errors)
------------------------------------------------------------
  STEP 1: called create_task with {'title': 'Renew SSL cert', 'priority': 'urgent', 'due_date': 'July 30th 2026'}
    -> Error: Invalid value for 'priority': got 'urgent'. Valid values: 'low', 'medium', 'high', 'critical'.
  STEP 2: called create_task with {'title': 'Renew SSL cert', 'due_date': 'July 30th 2026', 'priority': 'high'}
    -> Error: Invalid date format for 'due_date': got 'July 30th 2026'. Use ISO 8601 'YYYY-MM-DD' (e.g. '2026-07-30').
  STEP 3: called create_task with {'due_date': '2026-07-30', 'priority': 'high', 'title': 'Renew SSL cert'}
    -> {"status": "created", "due_date": "2026-07-30", "priority": "high", "titl

> The above blunt succeeds in less attempt/step than the instructive because the model was able to guess it correctly but that won't be the case always so we should always give good error messages when it fails

#### Same exercise but different example - more blunt and hard to guess

In [37]:
# A field with NO semantic link to the prompt -- there is exactly one valid
# value and nothing in the request hints at what it is.
PROVISION_SCHEMA = {
    "required": ["title", "team_code"],
    "team_code_exact": "SEC-7743",
}


def validate_provision_input(input_dict, schema):
    for field in schema["required"]:
        if field not in input_dict:
            return False, f"Missing required field '{field}'. Example: title='New DB instance'."
    if input_dict.get("team_code") != schema["team_code_exact"]:
        return False, (
            f"Invalid value for 'team_code': got {input_dict.get('team_code')!r}. "
            f"This request requires team_code={schema['team_code_exact']!r} exactly "
            f"-- it is an internal approval code, there is no other valid value."
        )
    return True, None


PROVISION_TOOL = [
    {
        "type": "function",
        "function": {
            "name": "provision_resource",
            "description": "Provision a new resource for a team. Requires a title and a team_code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "team_code": {"type": "string"},
                },
                "required": ["title", "team_code"],
            },
        },
    },
]


def _provision_instructive(**kwargs):
    valid, error = validate_provision_input(kwargs, PROVISION_SCHEMA)
    if not valid:
        return f"Error: {error}"
    return json.dumps({"status": "provisioned", **kwargs})


def _provision_blunt(**kwargs):
    valid, _error = validate_provision_input(kwargs, PROVISION_SCHEMA)
    if not valid:
        return "Error: 400 Bad Request"
    return json.dumps({"status": "provisioned", **kwargs})


DEMO3B_PROMPT = (
    "Provision a new database instance for the security team. "
    "Use the provision_resource tool."
)

print(f"Request (the correct team_code cannot be inferred from this text at all):")
print(f"  {DEMO3B_PROMPT}")

results_b = {}
for label, fn in [("instructive", _provision_instructive), ("blunt", _provision_blunt)]:
    print()
    print("-" * 60)
    print(f"UNGUESSABLE-VALUE DEMO: provision_resource ({label} errors)")
    print("-" * 60)
    results_b[label] = run_repair_loop(PROVISION_TOOL, {"provision_resource": fn}, DEMO3B_PROMPT, max_steps=4)

print()
print("-" * 60)
print("SUMMARY")
print("-" * 60)
for label, calls in results_b.items():
    succeeded = any("provisioned" in result for _n, _a, result in calls)
    print(f"  {label:12s} attempts={len(calls)} succeeded={succeeded}")
print()
print("How to read this:")
print("  * 'team_code' has no natural-language link to the prompt -- guessing it")
print("    right is essentially impossible without being told the exact value.")
print("  * The instructive run should succeed in 2 attempts: one rejection that")
print("    names the exact required code, then a correct retry.")
print("  * The blunt run should burn all 4 attempts and still fail -- 'Error: 400")
print("    Bad Request' gives it nothing to correct toward, so it can only")
print("    re-guess titles/codes at random, forever wrong.")


Request (the correct team_code cannot be inferred from this text at all):
  Provision a new database instance for the security team. Use the provision_resource tool.

------------------------------------------------------------
UNGUESSABLE-VALUE DEMO: provision_resource (instructive errors)
------------------------------------------------------------
  STEP 1: called provision_resource with {'team_code': 'security', 'title': 'database instance'}
    -> Error: Invalid value for 'team_code': got 'security'. This request requires team_code='SEC-7743' exactly -- it is an internal approval code, there is no other valid value.
  STEP 2: called provision_resource with {'team_code': 'SEC-7743', 'title': 'database instance'}
    -> {"status": "provisioned", "team_code": "SEC-7743", "title": "database instance"}
  STEP 3: no more tool calls -- final answer: The database instance has been successfully provisioned for the security team (team_code: SEC-7743).  
**Provisioned resource:**  
- Title: 

## Exercise 4 -- The Destructive Flag and the Gate That Reads It

Some tool calls only read data (`search_tasks`); others change something real
(`set_task_status`). The idea here is simple: mark the ones that change
something with a `destructive: true` flag on the schema, and write an
orchestrator that checks that flag *before* running the tool -- asking a human
first if it's set, and running straight through if it isn't.

The flag by itself does nothing. A model reading a schema doesn't stop itself
from calling a tool just because you wrote `destructive: true` somewhere --
that field means nothing to it. The only thing that makes it real is your own
code choosing to check it before dispatch. That's the whole exercise: build
the check, and prove a denial actually stops the mutation rather than just
getting logged and ignored.

One format note: Anthropic has an official `metadata` slot on a tool schema
for exactly this kind of flag. OpenAI/Ollama doesn't -- so `destructive` goes
at the top level, as a sibling of `function`, and it only means something
because your code and your schema privately agreed it would.

Implement `run_tool_with_gate(tool_name, arguments, schemas, dispatch,
approve_fn)` so that it:

1. finds the schema whose `function.name` matches `tool_name`
2. if that schema has `destructive: true`, calls `approve_fn(tool_name,
   arguments)` first, and if it returns `False`, refuses -- returning the string
   `"Action cancelled by human reviewer."` **without** executing anything
3. otherwise (non-destructive, or approved) executes
   `dispatch[tool_name](**arguments)` and returns its result

In [38]:
# The same three good tools, now carrying an explicit destructive flag.
# Note where the key sits: top level, a sibling of "function", NOT inside it.
GATED_TOOL_SCHEMAS = []
for _schema in GOOD_TOOL_SCHEMAS:
    _copy = json.loads(json.dumps(_schema))          # deep copy
    _copy["destructive"] = _copy["function"]["name"] == "set_task_status"
    GATED_TOOL_SCHEMAS.append(_copy)

print("destructive flags:")
for _s in GATED_TOOL_SCHEMAS:
    print(f"  {_s['function']['name']:20s} destructive={_s['destructive']}")


def run_tool_with_gate(tool_name, arguments, schemas, dispatch, approve_fn):
    """
    Execute a tool, but route anything flagged destructive through approve_fn first.

    Returns the tool's result string, or "Action cancelled by human reviewer."
    if approve_fn returned False. Never executes a destructive tool that was
    not approved.
    """
    # TODO 6: look up the schema in `schemas` by function.name == tool_name.
    # If it is missing, return an instructional error (Section 5 style).
    # If schema.get("destructive") is True, call approve_fn(tool_name, arguments)
    # and return "Action cancelled by human reviewer." when it returns False.
    # Otherwise call dispatch[tool_name](**arguments) and return the result.
    schema = next((s for s in schemas if s["function"]["name"] == tool_name), None)
    if schema is None:
        available = [s["function"]["name"] for s in schemas]
        return f"Error: no such tool '{tool_name}'. Available tools: {available}."

    if schema.get("destructive"):
        if not approve_fn(tool_name, arguments):
            return "Action cancelled by human reviewer."

    try:
        return dispatch[tool_name](**arguments)
    except TypeError as exc:
        # A wrong-arguments call is a Section 5 moment, not a crash.
        expected = list(schema["function"]["parameters"].get("properties", {}))
        return f"Error calling '{tool_name}': {exc}. Expected parameters: {expected}."

destructive flags:
  search_tasks         destructive=False
  get_task_detail      destructive=False
  set_task_status      destructive=True


In [39]:
approval_log = []


def approve_everything(tool_name, arguments):
    approval_log.append(("asked", tool_name))
    return True


def approve_nothing(tool_name, arguments):
    approval_log.append(("asked", tool_name))
    return False


# 1. A read-only tool must NOT trigger the gate at all.
approval_log.clear()
result = run_tool_with_gate(
    "search_tasks", {"assignee": "alex", "status": "open"},
    GATED_TOOL_SCHEMAS, GOOD_DISPATCH, approve_nothing,
)
assert approval_log == [], (
    f"search_tasks is not destructive -- approve_fn should never have been called, "
    f"but the log shows {approval_log}"
)
assert "task_" in result, f"search_tasks should have executed and returned matches, got {result!r}"

# 2. A destructive tool that is DENIED must not execute.
_status_before = next(t for t in TASK_STORE if t["id"] == "task_1")["status"]
approval_log.clear()
result = run_tool_with_gate(
    "set_task_status", {"task_id": "task_1", "new_status": "closed"},
    GATED_TOOL_SCHEMAS, GOOD_DISPATCH, approve_nothing,
)
assert result == "Action cancelled by human reviewer.", f"expected a refusal, got {result!r}"
assert approval_log == [("asked", "set_task_status")], "the gate should have asked exactly once"
_status_after = next(t for t in TASK_STORE if t["id"] == "task_1")["status"]
assert _status_after == _status_before, (
    f"THE GATE LEAKED: task_1 changed from {_status_before!r} to {_status_after!r} "
    f"even though approval was denied."
)

# 3. A destructive tool that is APPROVED must execute.
approval_log.clear()
result = run_tool_with_gate(
    "set_task_status", {"task_id": "task_1", "new_status": "closed"},
    GATED_TOOL_SCHEMAS, GOOD_DISPATCH, approve_everything,
)
assert approval_log == [("asked", "set_task_status")], "the gate should still have asked"
assert next(t for t in TASK_STORE if t["id"] == "task_1")["status"] == "closed", (
    "an approved destructive call should actually have taken effect"
)

# 4. An unknown tool gets an instructional error, not a KeyError.
result = run_tool_with_gate(
    "delete_everything", {}, GATED_TOOL_SCHEMAS, GOOD_DISPATCH, approve_everything,
)
assert "no such tool" in result.lower(), f"expected an instructional error, got {result!r}"
assert "search_tasks" in result, "the error should list what IS available (Section 5)"

_good_set_task_status("task_1", "open")  # restore the store for later cells

print("-" * 60)
print("PASS -- the gate asks only about destructive tools, and a denial")
print("       genuinely prevents the mutation rather than just logging it")
print("-" * 60)
print()
print("This is the hook Chapter 16 attaches to. Note what made it work: not the")
print("word 'destructive' appearing anywhere in a description the model reads,")
print("but a boolean your orchestrator checks BEFORE dispatch. A flag nothing")
print("reads is decoration; a flag read after execution is an audit log.")

------------------------------------------------------------
PASS -- the gate asks only about destructive tools, and a denial
       genuinely prevents the mutation rather than just logging it
------------------------------------------------------------

This is the hook Chapter 16 attaches to. Note what made it work: not the
word 'destructive' appearing anywhere in a description the model reads,
but a boolean your orchestrator checks BEFORE dispatch. A flag nothing
reads is decoration; a flag read after execution is an audit log.


### Live Demo -- Watch Your Gate Stand Between the Model and a Mutation

*Not graded; needs `ollama serve`.* The assert above proved the gate blocks an
unapproved destructive call. What it could not show is the part that matters
in a real agent: the model *wants* to close the task, emits a perfectly valid
`set_task_status` call, and your code is the only thing standing in the way.

The same request runs twice -- once with an approver that denies everything,
once with one that approves. Watch what the model does with a refusal: a good
agent reports it honestly rather than pretending the work is done (Chapter 1's
victory-declaration bias, live).

Each run gets its own deep copy of `TASK_STORE`, so this cell is safe to
re-run and does not depend on whether the assert cell above already mutated
the real store.


In [44]:
SET_STATUS_SCHEMA = [s for s in GATED_TOOL_SCHEMAS if s["function"]["name"] == "set_task_status"]

DEMO4_TARGET = "task_9"
DEMO4_PROMPT = (
    f"The security audit ({DEMO4_TARGET}) is finished. Close it, then tell me "
    f"exactly what you did."
)

if stub_filled("Exercise 4",
               lambda: run_tool_with_gate("search_tasks", {}, GATED_TOOL_SCHEMAS,
                                          GOOD_DISPATCH, lambda n, a: True)):

    # ============== RUN 1: the approver says NO ==============
    print("-" * 60)
    print("APPROVAL-GATE DEMO: DENIED")
    print("-" * 60)

    # a fresh copy of the store, just for this run
    store_denied = json.loads(json.dumps(TASK_STORE))

    def set_task_status_denied(task_id, new_status, **_):
        task = next(t for t in store_denied if t["id"] == task_id)
        task["status"] = new_status
        return json.dumps({"id": task_id, "status": new_status})

    # STEP 1: ask the model to close the task
    messages = [{"role": "user", "content": DEMO4_PROMPT}]
    response = client.chat.completions.create(model=OLLAMA_MODEL, tools=SET_STATUS_SCHEMA, messages=messages)
    msg = response.choices[0].message

    if not msg.tool_calls:
        print(f"  model didn't call the tool this run -- re-run the cell. It said: {msg.content}")
    else:
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments or "{}")
        print(f"  STEP 1: model called set_task_status with {args}")

        # STEP 2: the call goes through the gate -- approver returns False
        result = run_tool_with_gate(
            "set_task_status", args, SET_STATUS_SCHEMA,
            {"set_task_status": set_task_status_denied},
            approve_fn=lambda tool_name, arguments: False,
        )
        print(f"  STEP 2: gate result: {result}")

        # STEP 3: send the gate's result back so the model can react to it
        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        final = client.chat.completions.create(
            model=OLLAMA_MODEL, tools=SET_STATUS_SCHEMA, messages=messages, max_tokens=800,
        )
        final_msg = final.choices[0].message
        final_text = (final_msg.content or "").strip()
        if final_text:
            print(f"  STEP 3: model's closing words: {final_text}")
        else:
            print(f"  STEP 3: model returned no visible text this run "
                  f"(finish_reason={final.choices[0].finish_reason!r}) -- likely spent its "
                  f"whole budget 'thinking' with qwen3. Re-run the cell, or raise max_tokens further.")

        # STEP 4: did the task actually change?
        task = next(t for t in store_denied if t["id"] == DEMO4_TARGET)
        print(f"  STEP 4: {DEMO4_TARGET} status afterwards: {task['status']!r}")

    print()

    # ============== RUN 2: the approver says YES ==============
    print("-" * 60)
    print("APPROVAL-GATE DEMO: APPROVED")
    print("-" * 60)

    store_approved = json.loads(json.dumps(TASK_STORE))

    def set_task_status_approved(task_id, new_status, **_):
        task = next(t for t in store_approved if t["id"] == task_id)
        task["status"] = new_status
        return json.dumps({"id": task_id, "status": new_status})

    messages = [{"role": "user", "content": DEMO4_PROMPT}]
    response = client.chat.completions.create(model=OLLAMA_MODEL, tools=SET_STATUS_SCHEMA, messages=messages)
    msg = response.choices[0].message

    if not msg.tool_calls:
        print(f"  model didn't call the tool this run -- re-run the cell. It said: {msg.content}")
    else:
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments or "{}")
        print(f"  STEP 1: model called set_task_status with {args}")

        result = run_tool_with_gate(
            "set_task_status", args, SET_STATUS_SCHEMA,
            {"set_task_status": set_task_status_approved},
            approve_fn=lambda tool_name, arguments: True,
        )
        print(f"  STEP 2: gate result: {result}")

        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        final = client.chat.completions.create(
            model=OLLAMA_MODEL, tools=SET_STATUS_SCHEMA, messages=messages, max_tokens=800,
        )
        final_msg = final.choices[0].message
        final_text = (final_msg.content or "").strip()
        if final_text:
            print(f"  STEP 3: model's closing words: {final_text}")
        else:
            print(f"  STEP 3: model returned no visible text this run "
                  f"(finish_reason={final.choices[0].finish_reason!r}) -- likely spent its "
                  f"whole budget 'thinking' with qwen3. Re-run the cell, or raise max_tokens further.")

        task = next(t for t in store_approved if t["id"] == DEMO4_TARGET)
        print(f"  STEP 4: {DEMO4_TARGET} status afterwards: {task['status']!r}")

    print()
    print("How to read this:")
    print("  * Same prompt, same model, same tool schema in both runs.")
    print("  * DENIED: the model tried to close the task, the gate said no, and the")
    print("    status never changed.")
    print("  * APPROVED: the exact same kind of call went through, and the status")
    print("    DID change.")
    print("  * The only thing that differs between the two runs is approve_fn --")
    print("    that one function is the entire gate.")


------------------------------------------------------------
APPROVAL-GATE DEMO: DENIED
------------------------------------------------------------
  STEP 1: model called set_task_status with {'task_id': 'task_9', 'new_status': 'closed'}
  STEP 2: gate result: Action cancelled by human reviewer.
  STEP 3: model's closing words: The attempt to close task_9 was canceled by a human reviewer. This might indicate:

1. The task ID `task_9` doesn't exist in the tracker
2. The reviewer is enforcing a policy to confirm closure of security-related tasks
3. There's a validation rule preventing status changes without additional context

Would you like me to:
- Verify if task_9 exists?
- Provide alternative steps to close the task?
- Explain the reviewer's cancellation reason?
  STEP 4: task_9 status afterwards: 'open'

------------------------------------------------------------
APPROVAL-GATE DEMO: APPROVED
------------------------------------------------------------
  STEP 1: model called set_ta

### What This Demo Is Actually Showing

The one-sentence version: **the model can generate a perfectly valid, correctly-formed request to close the task -- and your code is the only thing that decides whether it's allowed to happen.**

#### The problem this solves

Nothing about a tool schema stops a model from calling it. `set_task_status` looks exactly as callable to the model whether you're about to approve it or not -- there's no red flag baked into the JSON that says "ask a human first." If your own code doesn't check for that before running the function, a destructive call just... runs, the moment the model decides to make it. This demo proves that check actually exists and actually works, using a real model instead of hand-written test data.

#### The flow, as a diagram

```text
 you                      model                       your gate                 the store
  |                          |                             |                         |
  |-- "close task_9" ------->|                             |                         |
  |                          |-- set_task_status(--------->|                         |
  |                          |   task_id="task_9",         |                         |
  |                          |   new_status="closed") -----|                         |
  |                          |                       [checks: is this tool           |
  |                          |                        marked destructive?]           |
  |                          |                       [yes -> ask approve_fn]         |
  |                          |                             |                         |
  |                          |                    DENIED run:  APPROVED run:         |
  |                          |                    approve_fn    approve_fn           |
  |                          |                    returns False returns True         |
  |                          |                             |                         |
  |                          |<-- "cancelled by  <-- OR -->|-- runs the real ----->  | status: open -> closed
  |                          |     reviewer" --------------|   function -----------> |
  |                          |                             |                         |
  |                    [reads the result,                                            |
  |                     writes a final answer]                                       |
  |<-- final answer ---------|                                                       |
```

Notice the model does the exact same thing in both runs -- same tool, same arguments. The branch only happens inside `run_tool_with_gate`, which the model never sees.

#### Walking through your actual run

```
DENIED run:
  STEP 1: model asks to set task_9 -> "closed"
  STEP 2: gate checks destructive=True, calls approve_fn -> False
          gate returns "Action cancelled by human reviewer." WITHOUT
          ever calling the real set_task_status function
  STEP 3: model sees that rejection text and reports it
  STEP 4: task_9 status: still 'open'   <- the real function never ran

APPROVED run:
  STEP 1: model asks to set task_9 -> "closed"  (identical to above)
  STEP 2: gate checks destructive=True, calls approve_fn -> True
          gate now DOES call the real set_task_status function
  STEP 3: model reports success
  STEP 4: task_9 status: now 'closed'
```

The only line of code that differs between the two runs is `approve_fn` itself (`lambda tool_name, arguments: False` vs. `... True`). Everything else -- prompt, schema, model, the tool function that would have run -- is identical.

#### A subtlety worth noticing in your own output

Read the DENIED run's closing words again: the model correctly reports that the task was *not* closed -- that part is honest, and it's the behavior Section 7 cares about (no victory-declaration lie). But look closer at the reasons it offers ("maybe task_9 doesn't exist," "maybe there's a policy," "maybe a validation rule") -- none of that came from the gate. The gate only ever said `"Action cancelled by human reviewer."` The model doesn't know *why* it was cancelled, so it invents plausible-sounding explanations to fill the gap. That's a separate, smaller failure mode from the one this exercise targets: being honest about *what* happened is not the same as being honest about *why* -- the model will confidently fabricate a reason if you don't give it one.

#### Connection forward

This is the same "boolean your code checks before doing something irreversible" pattern you'll see again as a full approval workflow in later chapters -- the difference there is usually a human actually reading the pending action before pressing approve/deny, rather than `approve_fn` deciding instantly.

## Measuring the Token Bill: Bad vs Good

Section 8's arithmetic priced a 40-tool library. This is the same arithmetic at
a scale you can verify by hand: what does the bad tool set cost versus the good
one, on the exact same question?

The cell below deliberately reports a number that makes the good tool set look
*bad* -- its schemas are several times larger, because good descriptions are made
of tokens like everything else. That number is real and worth confronting rather
than skipping past.

What rescues it is a distinction Section 8 and Chapter 2 make together, and which
this cell measures explicitly: **tool schemas are a fixed prefix, tool results
are a growing tail.** The schema is the same bytes every turn, so it caches; a
tool result added on turn 3 is re-sent on turns 4 through 20, uncacheable, so its
cost compounds quadratically across the run. Trading a flat premium for a
quadratic saving is the actual deal on offer, and it only looks close if you
forget which of the two accumulates.

`estimate_tokens` is a deliberately crude `len(text) // 4` approximation, not a
real tokenizer -- the point is the ratio, which is robust to that imprecision.

In [10]:
def estimate_tokens(text):
    """Rough len(text)//4 approximation -- NOT a real tokenizer."""
    return max(1, len(text) // 4)


# --- Part 1: the RETURN VALUE bill (Section 4) -----------------------------
# Same question -- "alex's open high-priority tasks" -- answered by each tool set.
bad_response = _bad_dump_tasks()   # no filters exist, so the model gets everything
good_response = _good_search_tasks(assignee="alex", status="open", priority="high")

bad_tokens = estimate_tokens(bad_response)
good_tokens = estimate_tokens(good_response)

print("-" * 60)
print("RETURN VALUE: dump_tasks() [bad] vs search_tasks(...) [good]")
print("-" * 60)
print(f"  bad  : {bad_tokens:5d} est. tokens  ({len(bad_response)} chars, all {len(TASK_STORE)} records, every field)")
print(f"  good : {good_tokens:5d} est. tokens  ({len(good_response)} chars, {len(GROUND_TRUTH_ALEX_OPEN_HIGH)} matching summaries)")
print(f"  -> {(1 - good_tokens / bad_tokens) * 100:.0f}% reduction on ONE call")
print()
print("  And per Chapter 2, this cost is not paid once: every tool result stays in")
print("  the transcript and is re-sent on every subsequent turn. Part 3 below works")
print("  out how many times over that actually is -- it is more than you'd guess.")

# --- Part 2: the SCHEMA bill (Section 8) -----------------------------------
bad_schema_tokens = estimate_tokens(json.dumps(BAD_TOOL_SCHEMAS))
good_schema_tokens = estimate_tokens(json.dumps(GOOD_TOOL_SCHEMAS))

print()
print("-" * 60)
print("SCHEMA SIZE: the bill you pay BEFORE any tool is called")
print("-" * 60)
print(f"  bad  tool schemas: {bad_schema_tokens:5d} est. tokens")
print(f"  good tool schemas: {good_schema_tokens:5d} est. tokens")
print(f"  -> the good set costs {good_schema_tokens / bad_schema_tokens:.1f}x MORE to declare")
print()
print("  This inversion is the honest part of the trade, and it is the whole")
print("  reason this cell exists: good descriptions are NOT free. Anyone who")
print("  tells you better tool design is a pure win is skipping this number.")

# --- Part 3: why the schema cost loses anyway (Section 8 + Chapter 2) ------
# Two different cost curves are at work, and conflating them is what makes
# the trade look close when it is not:
#   * the schema is a FIXED PREFIX -- same bytes every turn, so it caches
#   * tool results land in the GROWING TAIL -- turn N re-sends turns 1..N-1
STEPS = 20

# Flat: schema tokens are re-sent every turn (identical bytes -> cacheable).
bad_schema_total = STEPS * bad_schema_tokens
good_schema_total = STEPS * good_schema_tokens

# Quadratic: a result added on turn k is re-sent on every later turn, so the
# run pays sum(k for k in 1..STEPS) = STEPS*(STEPS+1)/2 copies of each result.
ACCUMULATION_FACTOR = STEPS * (STEPS + 1) // 2   # 210 for STEPS=20
bad_result_total = ACCUMULATION_FACTOR * bad_tokens
good_result_total = ACCUMULATION_FACTOR * good_tokens

print()
print("-" * 60)
print(f"THE {STEPS}-STEP BILL: fixed prefix vs growing tail")
print("-" * 60)
print(f"  A result added on turn k is re-sent on every turn after it, so the run")
print(f"  pays {ACCUMULATION_FACTOR}x each result ({STEPS}x{STEPS + 1}/2), not {STEPS}x. Schemas stay flat at {STEPS}x.")
print()
print(f"  {'':6s} {'schema (flat)':>16s} {'results (accum.)':>18s} {'total':>10s}")
for label, sch, res in [("bad ", bad_schema_total, bad_result_total),
                        ("good", good_schema_total, good_result_total)]:
    print(f"  {label:6s} {sch:16,d} {res:18,d} {sch + res:10,d}")

_bad_total = bad_schema_total + bad_result_total
_good_total = good_schema_total + good_result_total
print()
print(f"  -> the good set costs {good_schema_total - bad_schema_total:+,d} tokens more in schemas")
print(f"     and {good_result_total - bad_result_total:+,d} tokens less in results")
print(f"  -> net: {(1 - _good_total / _bad_total) * 100:.0f}% cheaper overall ({_bad_total:,} -> {_good_total:,})")
print()
print("  That is the actual shape of the trade. You pay a fixed, cacheable")
print("  premium once per turn for descriptions, to avoid a cost that compounds")
print("  quadratically in the part of the context nothing can cache. Notice how")
print("  badly a naive comparison misleads here: charge each result only once")
print(f"  per turn instead of {ACCUMULATION_FACTOR}x and the bad set looks CHEAPER, which is exactly")
print("  the mistake that gets verbose tool returns shipped.")

------------------------------------------------------------
RETURN VALUE: dump_tasks() [bad] vs search_tasks(...) [good]
------------------------------------------------------------
  bad  :   464 est. tokens  (1857 chars, all 12 records, every field)
  good :    48 est. tokens  (194 chars, 2 matching summaries)
  -> 90% reduction on ONE call

  And per Chapter 2, this cost is not paid once: every tool result stays in
  the transcript and is re-sent on every subsequent turn. Part 3 below works
  out how many times over that actually is -- it is more than you'd guess.

------------------------------------------------------------
SCHEMA SIZE: the bill you pay BEFORE any tool is called
------------------------------------------------------------
  bad  tool schemas:   146 est. tokens
  good tool schemas:   586 est. tokens
  -> the good set costs 4.0x MORE to declare

  This inversion is the honest part of the trade, and it is the whole
  reason this cell exists: good descriptions are NOT

### What This Cell Is Actually Showing

The one-sentence version: **good tool design costs more tokens to declare, but saves far more tokens than that over an actual conversation -- and this cell proves both halves with real numbers instead of asking you to take it on faith.**

#### Part 1 -- the answer itself is smaller

`dump_tasks()` (bad) hands back every field of every task, because it has no filters. `search_tasks(...)` (good) hands back only the rows that actually match, already summarized. Same question, two very different answers -- one of them padded with everything the model didn't ask for. That padding isn't free: every extra character becomes tokens the model has to read before it can even start answering.

#### Part 2 -- the honest, uncomfortable part

Here's the twist the cell deliberately doesn't hide: the *good* tool schemas are bigger than the bad ones, in tokens. Why? Because "Gets stuff" is three tokens and a real, useful description like "Search tasks by assignee, status, and priority; returns up to N summaries per call" is a lot more than three tokens. Good descriptions aren't free -- they're the whole reason the model can use the tool correctly, and that clarity costs bytes. If you stopped reading here, it would look like bad tool design wins.

#### Part 3 -- why the schema cost loses anyway

This is the part that actually settles it, and the key idea is: **a schema and a tool result age completely differently over a conversation.**

Think of the schema like a rulebook you hand someone once. Every turn, you hand them the *exact same* rulebook again -- since it never changes, a smart system can cache it and barely pay for it twice. So across a 20-turn conversation, the schema cost just adds up flatly: 20 x (schema size).

A tool *result*, on the other hand, is like a receipt that gets stapled into a growing case file. Once a result lands in the conversation on turn 3, it doesn't just get read once -- the *entire* conversation so far (including that receipt) gets resent to the model on turn 4, and turn 5, and every turn after that, because the model has no memory between calls except what's in the message list. A result from turn 3 in a 20-turn conversation gets re-read roughly 17 more times.

```text
Turn:            1    2    3    4    5   ...   20
Schema paid:      X    X    X    X    X   ...    X      <- flat, same X every time
Result from       .    .    X -> X -> X -> ... -> X     <- once added, re-sent
turn 3 paid:                                              on every later turn too
```

Add that up across every turn's result and you get a triangular number (`1+2+3+...+20`), not a flat `20x` -- that's the `ACCUMULATION_FACTOR` in the code. A small result difference gets multiplied by that much larger factor, while the schema difference only ever gets multiplied by the turn count. That's why the "good schema costs more" fact from Part 2 stops mattering once a real multi-turn conversation is in play -- you pay the bigger cost once per turn, to avoid a smaller-looking cost that actually compounds.

#### The gotcha worth remembering

If you ever compare tool designs by charging each tool result only once (instead of once per remaining turn), the bad set can look cheaper -- and that miscount is exactly how verbose, unfiltered tool results end up shipped to production. The growing-tail cost is the one that's easy to forget because it's invisible in a single-turn test.


## Setting Up an Experiment That Can Actually Fail

Everything above was verifiable without a model, which is why it was worth doing
first. This section is the part that cannot be faked: the same question, the same
local model, two tool sets that differ only in *interface* quality.

Before running it, the conditions have to be right, and this is worth being
explicit about because getting it wrong produces a confident null result. The
first version of this notebook ran the comparison with **3 tools over the
12-task store** and found *no difference whatsoever* -- both sets answered
correctly in the same number of steps, and the bad set was marginally cheaper.

That was a badly designed experiment, not evidence against Section 9. Re-read
what Section 9 actually claims and you can see why it could not possibly have
shown up:

| Section 9's stated condition | The 3-tool/12-task setup | Fixed below by |
|---|---|---|
| degradation appears in the **20–50 tool** range | 3 tools | padding both sets to ~20 tools |
| worse when tools **overlap in name/purpose** | 3 clearly distinct tools | bad set gets `search`/`find`/`lookup`/`query` |
| the model must do work the tool could have done | 12 records is trivial to filter in-context | a 60-task store |

There is also a subtler flaw: the question ("alex's open high-priority tasks")
was answerable by dumping everything and filtering in-context, which an 8B model
does perfectly at 12 records. The question below asks for **overdue** tasks
instead -- open *and* past a reference date. The good schema documents
`status='overdue'` and computes it server-side; the bad set offers no way to
express it, so the model must do date arithmetic across 60 records by hand. That
is where the interface difference stops being cosmetic.

None of this is stacking the deck. It is reproducing the conditions the notes
name, which the first attempt simply did not have.

### One Variable at a Time (and a Real Finding Found by Accident)

A second version of this experiment kept the 5-row page size from the 12-task
store. With 10 correct answers, that meant a complete answer required the model
to read `next_cursor` and call `search_tasks` a second time -- and on one attempt
it simply did not, answering confidently with the first 5 of 10.

That is a genuine and useful observation, and it is worth stating plainly rather
than burying: **a well-designed continuation handle still depends on the model
choosing to follow it.** Section 4 gets you an honest tool; it does not get you
an obedient reader. That is a loop-level concern, which is exactly why Section 12
lists verification as Chapter 6's job rather than something tool design can fix.

But it cannot be allowed to *contaminate this measurement*, because a wrong
answer would then be ambiguous between two entirely different causes:

```text
[ WHY ONE MEASUREMENT CANNOT ANSWER TWO QUESTIONS ]

  wrong final answer
        |
        +--> did the model pick the wrong tool?        <- Section 9, what we measure
        |
        +--> did it pick the right tool and then       <- Section 4, already tested
             ignore the continuation handle?              in Exercise 2

  With a 5-row page over 10 answers, BOTH paths are open, so the number
  reported means neither thing. With a page that holds the whole answer,
  only the first path remains -- and the result becomes interpretable.
```

So the experiment below uses `BIG_PAGE_SIZE = 25`, large enough to hold the whole
answer in one call, and the `search_tasks` description is **edited to say so**.
Copying the schema while leaving it claiming "up to 5" would have made the schema
lie about its own behaviour -- Section 2's cardinal sin, and a particularly silly
one to commit inside this chapter's own experiment. There is an `assert` below
enforcing that the promise and the behaviour still agree.

In [11]:
# The 12-task store above was deliberately small enough to verify by eye.
# Section 9's effect needs realistic scale, so build a bigger one -- generated
# deterministically (no random seed) so every run grades against the same truth.
_TITLES = ["Fix login bug", "Update docs", "Migrate database", "Design logo",
           "Refactor auth", "Onboarding guide", "Fix payment bug", "Q3 roadmap",
           "Security audit", "Bump deps", "Interview notes", "Rate limiter"]
_PEOPLE = ["alex", "priya", "sam", "jordan"]
_PRIORITIES = ["low", "medium", "high", "critical"]
_PROJECTS = ["auth", "docs", "infra", "billing", "planning", "research"]

BIG_TASK_STORE = []
for _n in range(60):
    # Spread due dates either side of REFERENCE_DATE (2026-08-01) so "overdue"
    # is a real distinction rather than all-or-nothing.
    _month, _day = (7, (_n % 28) + 1) if _n % 3 else (8, (_n % 27) + 2)
    BIG_TASK_STORE.append({
        "id": f"task_{_n + 1}",
        "title": f"{_TITLES[_n % len(_TITLES)]} #{_n + 1}",
        "assignee": _PEOPLE[_n % len(_PEOPLE)],
        "status": "closed" if _n % 4 == 3 else "open",
        "priority": _PRIORITIES[_n % len(_PRIORITIES)],
        "due_date": f"2026-{_month:02d}-{_day:02d}",
        "project": _PROJECTS[_n % len(_PROJECTS)],
    })

TARGET_PERSON = "sam"
GROUND_TRUTH_OVERDUE = [
    t for t in BIG_TASK_STORE
    if t["assignee"] == TARGET_PERSON and t["status"] == "open" and t["due_date"] < REFERENCE_DATE
]

print(f"BIG_TASK_STORE: {len(BIG_TASK_STORE)} tasks")
print(f"  open: {sum(1 for t in BIG_TASK_STORE if t['status'] == 'open')}, "
      f"closed: {sum(1 for t in BIG_TASK_STORE if t['status'] == 'closed')}")
print(f"  overdue overall: {sum(1 for t in BIG_TASK_STORE if t['status'] == 'open' and t['due_date'] < REFERENCE_DATE)}")
print(f"\nThe question: which of {TARGET_PERSON}'s tasks are OVERDUE "
      f"(open AND due before {REFERENCE_DATE})?")
print(f"Ground truth: {len(GROUND_TRUTH_OVERDUE)} task(s)")
for _t in GROUND_TRUTH_OVERDUE:
    print(f"  {_t['id']}: {_t['title']} (due {_t['due_date']})")

print(f"\nWhat each tool set makes the model do:")
_dump = json.dumps(BIG_TASK_STORE)
print(f"  BAD : dump_tasks() returns all {len(BIG_TASK_STORE)} records "
      f"({len(_dump)} chars, ~{len(_dump) // 4} est. tokens), and the model must")
print(f"        compare 60 due-dates against {REFERENCE_DATE} in its head.")
print(f"  GOOD: search_tasks(assignee='{TARGET_PERSON}', status='overdue') returns "
      f"{len(GROUND_TRUTH_OVERDUE)} row(s), computed server-side.")


# --- The two tool sets, rebuilt over BIG_TASK_STORE and padded to ~20 tools ---
def _big_bad_dump_tasks():
    return json.dumps(BIG_TASK_STORE)


def _big_bad_do_thing(x=None, opts=None):
    opts = opts or {}
    meta = opts.get("filters", {}).get("meta", {})
    who, state = meta.get("who"), meta.get("state")
    return json.dumps([
        t for t in BIG_TASK_STORE
        if (who is None or t["assignee"] == who) and (state is None or t["status"] == state)
    ])


# Deliberately larger than the 12-task store's PAGE_SIZE=5. Rationale below in
# the markdown: with 10 correct answers and a 5-row page, a wrong final answer
# could mean EITHER "picked the wrong tool" (Section 9, what we're measuring)
# OR "ignored the continuation handle" (Section 4, already tested in Exercise 2).
# One measurement cannot answer two questions, so this page holds the full answer.
BIG_PAGE_SIZE = 25


def _big_good_search_tasks(assignee=None, status=None, priority=None, cursor=0,
                           response_format="concise"):
    def status_matches(task):
        if status is None:
            return True
        if status == "overdue":
            return task["status"] == "open" and task["due_date"] < REFERENCE_DATE
        return task["status"] == status

    matches = [
        t for t in BIG_TASK_STORE
        if (assignee is None or t["assignee"] == assignee)
        and status_matches(t)
        and (priority is None or t["priority"] == priority)
    ]
    page = matches[cursor: cursor + BIG_PAGE_SIZE]
    remaining = len(matches) - (cursor + len(page))
    results = page if response_format == "detailed" else [
        f"{t['id']}: {t['title']} ({t['priority']}, due {t['due_date']})" for t in page
    ]
    payload = {"results": results, "total_matches": len(matches)}
    if remaining > 0:
        payload["next_cursor"] = cursor + len(page)
        payload["message"] = (f"Showing {len(page)} of {len(matches)}. {remaining} remain -- "
                              f"call again with cursor={cursor + len(page)}.")
    else:
        payload["next_cursor"] = None
        payload["message"] = f"All {len(matches)} match(es) shown."
    return json.dumps(payload)


def _patched_search_schema():
    """
    The search_tasks schema with its stated page size corrected to match
    BIG_PAGE_SIZE. Copying the schema and leaving the description claiming
    'up to 5' would make the schema lie about its own behaviour -- the exact
    failure this whole chapter argues against, and it would be especially
    absurd to ship it inside the chapter's own experiment.
    """
    schema = json.loads(json.dumps(GOOD_TOOL_SCHEMAS[0]))
    schema["function"]["description"] = schema["function"]["description"].replace(
        "Returns up to 5 matches per call",
        f"Returns up to {BIG_PAGE_SIZE} matches per call",
    )
    return schema


def _stub_tool(**kwargs):
    """Every padding tool returns an empty result -- they exist to be *chosen*, not used."""
    return json.dumps({"results": [], "message": "No matching records in this system."})


def _fn_schema(name, description, props, required=None, extra=None):
    schema = {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": props,
                       **({"required": required} if required else {})}}}
    if extra:
        schema.update(extra)
    return schema


# BAD padding: overlapping, vague names -- Section 9's "semantic confusion".
# Which of these would YOU pick for "sam's overdue tasks"? That ambiguity is
# the entire failure mode, reproduced deliberately.
_BAD_PAD_NAMES = ["search", "find", "lookup", "query", "get_items", "list_all",
                  "fetch_data", "retrieve", "get_records", "filter_items",
                  "search_all", "find_items", "get_stuff", "list_data",
                  "query_items", "fetch_all"]
BIG_BAD_SCHEMAS = [
    _fn_schema("do_thing", "Does the thing.",
               {"x": {"type": "string"},
                "opts": {"type": "object", "properties": {"filters": {"type": "object"}}}}),
    _fn_schema("dump_tasks", "Dumps tasks.", {}),
    _fn_schema("t_upd", "Updates t.",
               {"i": {"type": "string"}, "s": {"type": "string"}}, ["i", "s"]),
] + [_fn_schema(n, "Searches records.", {"q": {"type": "string"}}) for n in _BAD_PAD_NAMES]

BIG_BAD_DISPATCH = {"do_thing": _big_bad_do_thing, "dump_tasks": _big_bad_dump_tasks,
                    "t_upd": lambda **kw: json.dumps({"ok": True})}
BIG_BAD_DISPATCH.update({n: _stub_tool for n in _BAD_PAD_NAMES})

# GOOD padding: namespaced by system, so nothing competes with search_tasks.
_GOOD_PAD = [
    ("calendar_list_events", "List calendar events in a date range. Returns event titles and times. Does not cover tasks -- use search_tasks for those."),
    ("calendar_create_event", "Create a calendar event. Mutates the calendar."),
    ("wiki_search_pages", "Full-text search over internal wiki pages. Returns page titles and excerpts, not tasks."),
    ("wiki_get_page", "Fetch one wiki page's full text by its slug."),
    ("crm_search_customers", "Search CRM customer records by name or email. Customers, not tasks."),
    ("crm_get_customer", "Fetch one CRM customer record by customer id."),
    ("github_list_pull_requests", "List open GitHub pull requests for a repository. Code review items, not tracker tasks."),
    ("github_get_pull_request", "Fetch one GitHub pull request by number, including its review state."),
    ("slack_search_messages", "Search Slack message history by keyword and channel."),
    ("slack_post_message", "Post a message to a Slack channel. This is visible to other people."),
    ("metrics_query_timeseries", "Query a named metric over a time range. Returns numeric datapoints, never task records."),
    ("metrics_list_dashboards", "List available metrics dashboards by name."),
    ("oncall_get_current_rotation", "Return who is currently on call for a given service."),
    ("oncall_page_engineer", "Page the on-call engineer. This wakes a human up -- destructive."),
    ("expenses_search_reports", "Search submitted expense reports by employee and date range."),
    ("expenses_approve_report", "Approve one expense report. Mutates finance records."),
]
BIG_GOOD_SCHEMAS = [
    _patched_search_schema(),                       # search_tasks -- the right answer
    json.loads(json.dumps(GOOD_TOOL_SCHEMAS[1])),   # get_task_detail
    json.loads(json.dumps(GOOD_TOOL_SCHEMAS[2])),   # set_task_status
] + [_fn_schema(n, d, {"search_query": {"type": "string", "description": "What to look for."}})
     for n, d in _GOOD_PAD]

# Guard: the schema's promise and the function's behaviour must agree.
assert f"up to {BIG_PAGE_SIZE} matches" in BIG_GOOD_SCHEMAS[0]["function"]["description"]
_probe = json.loads(_big_good_search_tasks(assignee=TARGET_PERSON, status="overdue"))
assert _probe["next_cursor"] is None, (
    f"The experiment's page must hold the whole answer, or a wrong result becomes "
    f"ambiguous between tool selection and pagination. Got next_cursor="
    f"{_probe['next_cursor']} for {len(GROUND_TRUTH_OVERDUE)} matches."
)
print(f"Page-size guard: one call returns all {_probe['total_matches']} matches "
      f"(next_cursor={_probe['next_cursor']}), so a wrong answer can only mean "
      f"wrong tool.")

BIG_GOOD_DISPATCH = {
    "search_tasks": _big_good_search_tasks,
    "get_task_detail": _good_get_task_detail,
    "set_task_status": _good_set_task_status,
}
BIG_GOOD_DISPATCH.update({n: _stub_tool for n, _ in _GOOD_PAD})

print()
print(f"BAD  library: {len(BIG_BAD_SCHEMAS)} tools  "
      f"({estimate_tokens(json.dumps(BIG_BAD_SCHEMAS))} est. schema tokens)")
print(f"  names: {[s['function']['name'] for s in BIG_BAD_SCHEMAS][:8]} ...")
print(f"GOOD library: {len(BIG_GOOD_SCHEMAS)} tools  "
      f"({estimate_tokens(json.dumps(BIG_GOOD_SCHEMAS))} est. schema tokens)")
print(f"  names: {[s['function']['name'] for s in BIG_GOOD_SCHEMAS][:8]} ...")
print()
print("Both libraries are now in Section 9's 20-tool range. Note that the GOOD")
print("library is the LARGER one in tokens -- it is not winning by being smaller.")

BIG_TASK_STORE: 60 tasks
  open: 45, closed: 15
  overdue overall: 30

The question: which of sam's tasks are OVERDUE (open AND due before 2026-08-01)?
Ground truth: 10 task(s)
  task_3: Migrate database #3 (due 2026-07-03)
  task_11: Interview notes #11 (due 2026-07-11)
  task_15: Migrate database #15 (due 2026-07-15)
  task_23: Interview notes #23 (due 2026-07-23)
  task_27: Migrate database #27 (due 2026-07-27)
  task_35: Interview notes #35 (due 2026-07-07)
  task_39: Migrate database #39 (due 2026-07-11)
  task_47: Interview notes #47 (due 2026-07-19)
  task_51: Migrate database #51 (due 2026-07-23)
  task_59: Interview notes #59 (due 2026-07-03)

What each tool set makes the model do:
  BAD : dump_tasks() returns all 60 records (9352 chars, ~2338 est. tokens), and the model must
        compare 60 due-dates against 2026-08-01 in its head.
  GOOD: search_tasks(assignee='sam', status='overdue') returns 10 row(s), computed server-side.
Page-size guard: one call returns all 10 matche

### What This Cell Is Actually Showing

The one-sentence version: **the earlier bad-vs-good comparison could not have shown a real difference, so this cell rebuilds the whole experiment at a scale where a difference actually has room to appear.**

#### Why the first attempt was doomed before it started

Section 9's claim (tool selection gets worse as your tool library grows and tools start looking alike) has three specific conditions attached to it: it shows up in the 20-50 tool range, it gets worse when tool names/purposes overlap, and it gets worse when the model has to do filtering work the tool could have done itself. The original setup had 3 clearly-different tools and 12 easy-to-scan records -- none of those three conditions were present, so of course no difference showed up. That wasn't proof the chapter's claim is wrong; it was proof the test never gave the claim a chance to fail.

#### What this cell actually builds

Three things, each aimed at one of those missing conditions:

1. **`BIG_TASK_STORE`** -- 60 tasks instead of 12, so filtering 60 due-dates in the model's head is a real burden instead of a glance.
2. **`BIG_BAD_SCHEMAS`** -- the original three bad tools, plus 16 padding tools with names like `search`, `find`, `lookup`, `query`, `fetch_data` -- all near-synonyms of each other. If you can't quickly tell which of those you'd call for "sam's overdue tasks," neither can the model; that ambiguity is Section 9's "semantic confusion" made concrete.
3. **`BIG_GOOD_SCHEMAS`** -- the original three good tools, plus 16 padding tools that are clearly labeled as belonging to *other systems* (`calendar_list_events`, `wiki_search_pages`, `crm_get_customer`...). Nothing here competes with `search_tasks` for "which tool handles a task question" -- there's exactly one obvious answer.

Both libraries land around 19-20 tools, which is Section 9's actual danger zone -- not a number picked for convenience.

#### The one deliberate design choice worth understanding: `BIG_PAGE_SIZE = 25`

The smaller store's page size was 5. Here it's bumped to 25, on purpose, so that one call returns *every* matching row. Why does that matter? Because if a wrong final answer could be explained by either "picked the wrong tool" (Section 9 -- what this experiment measures) *or* "picked the right tool but ignored the pagination hint" (Section 4 -- already proven separately in Exercise 2), a wrong answer here would be ambiguous about which failure actually happened. Forcing one call to hold the whole answer removes that second possibility entirely, so any wrong answer can only mean one thing: tool selection failed. The `assert` right after builds this guarantee and checks it rather than assuming it.

#### What to expect when you read the printed output

You'll see both libraries' tool names and their token counts -- and the good library should come out *larger* in tokens here too, same as the previous cell. That's intentional and consistent: this cell is only about proving the playing field is now fair (real scale, real overlap, real filtering burden). The actual bad-vs-good outcome is what the next cell (the live run) measures.


### The Run

The loop below is Chapter 2's cycle stripped to what this measurement needs. It
records five things per attempt: did it answer correctly, how many steps, how
many tool errors, how many tokens, and -- new, and the most direct test of
Section 9 -- **which tool it picked first.**

A caveat that stays true no matter how the numbers land: an 8B local model is
noisy, and `ATTEMPTS_PER_TOOL_SET` runs of each set is a small sample. What
should hold up is the *direction* and the *mechanism*, not the exact figures.

In [13]:
ATTEMPTS_PER_TOOL_SET = 3
REAL_PROMPT = (
    f"Which of {TARGET_PERSON}'s tasks are overdue? A task is overdue if it is "
    f"still open and its due date is before {REFERENCE_DATE}. "
    f"List their titles. Use the tools -- do not guess."
)


def run_loop(client, model, schemas, dispatch, prompt, max_steps=6):
    """
    Chapter 2's loop, trimmed to what this comparison measures.
    Returns dict: final_text, steps, tool_errors, tokens, tools_called.
    """
    messages = [{"role": "user", "content": prompt}]
    tokens = 0
    tool_errors = 0
    tools_called = []

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=model, tools=schemas, messages=messages)
        if response.usage:
            tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        assistant_msg = {"role": "assistant"}
        if msg.content:
            assistant_msg["content"] = msg.content
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": tc.type,
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if finish_reason == "stop" or not msg.tool_calls:
            return {"final_text": msg.content, "steps": step, "tool_errors": tool_errors,
                    "tokens": tokens, "tools_called": tools_called}

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            tools_called.append(name)
            try:
                args = json.loads(tool_call.function.arguments or "{}")
                fn = dispatch.get(name)
                if fn is None:
                    content = f"Error: no such tool '{name}'. Available: {list(dispatch)}."
                    tool_errors += 1
                else:
                    content = str(fn(**args))
                    if content.startswith("Error"):
                        tool_errors += 1
            except Exception as exc:
                content = f"Error: {type(exc).__name__}: {exc}"
                tool_errors += 1
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})

    return {"final_text": None, "steps": max_steps, "tool_errors": tool_errors,
            "tokens": tokens, "tools_called": tools_called}


def grade(final_text):
    """
    Did the answer name every overdue task and no extras?
    Titles carry a '#N' suffix, so a substring check is unambiguous here.
    """
    if not final_text:
        return False
    lowered = final_text.lower()
    found_all = all(t["title"].lower() in lowered for t in GROUND_TRUTH_OVERDUE)
    # Catch the classic in-context-filtering failure: naming tasks that are
    # NOT overdue (wrong person, closed, or due after the reference date).
    wrong = [t for t in BIG_TASK_STORE
             if t not in GROUND_TRUTH_OVERDUE and t["title"].lower() in lowered]
    return found_all and not wrong


expected_titles = [t["title"] for t in GROUND_TRUTH_OVERDUE]
print(f"Question: {REAL_PROMPT}")
print(f"Correct answer ({len(expected_titles)} task(s)): {expected_titles}")
print(f"Model: {OLLAMA_MODEL}  |  {ATTEMPTS_PER_TOOL_SET} attempt(s) per tool set")
print(f"Library size: BAD={len(BIG_BAD_SCHEMAS)} tools, GOOD={len(BIG_GOOD_SCHEMAS)} tools")
print()

results = {}
for set_name, schemas, dispatch in [("BAD", BIG_BAD_SCHEMAS, BIG_BAD_DISPATCH),
                                    ("GOOD", BIG_GOOD_SCHEMAS, BIG_GOOD_DISPATCH)]:
    print("-" * 68)
    print(f"{set_name} TOOL SET ({len(schemas)} tools)")
    print("-" * 68)
    runs = []
    for attempt in range(1, ATTEMPTS_PER_TOOL_SET + 1):
        try:
            outcome = run_loop(client, OLLAMA_MODEL, schemas, dispatch, REAL_PROMPT)
        except Exception as exc:
            print(f"  attempt {attempt}: request failed -- {type(exc).__name__}: {exc}")
            print("  (is 'ollama serve' running?)")
            continue
        outcome["correct"] = grade(outcome["final_text"])
        outcome["first_tool"] = outcome["tools_called"][0] if outcome["tools_called"] else None
        runs.append(outcome)
        print(f"  attempt {attempt}: correct={str(outcome['correct']):5s} "
              f"steps={outcome['steps']} tool_errors={outcome['tool_errors']} "
              f"tokens={outcome['tokens']}")
        print(f"    first tool chosen: {outcome['first_tool'] or '(none -- the model guessed)'}")
        print(f"    all tools called : {outcome['tools_called'] or '[]'}")
        snippet = (outcome["final_text"] or "(no final answer -- hit max_steps)").replace("\n", " ")
        print(f"    answer: {snippet[:180]}")
    results[set_name] = runs

print()
print("-" * 68)
print("SUMMARY")
print("-" * 68)
print(f"  {'tool set':10s} {'correct':>9s} {'avg steps':>10s} {'avg errors':>11s} {'avg tokens':>11s}")
for set_name, runs in results.items():
    if not runs:
        print(f"  {set_name:10s}    (no successful runs -- check the Ollama connection)")
        continue
    n = len(runs)
    print(f"  {set_name:10s} {sum(r['correct'] for r in runs)}/{n:<7d} "
          f"{sum(r['steps'] for r in runs) / n:10.1f} "
          f"{sum(r['tool_errors'] for r in runs) / n:11.1f} "
          f"{sum(r['tokens'] for r in runs) / n:11.0f}")

print()
print("  first-tool choice (Section 9's selection accuracy, most directly):")
for set_name, runs in results.items():
    if runs:
        print(f"    {set_name:5s}: {[r['first_tool'] for r in runs]}")

print()
print("How to read this, including the case where it does NOT come out cleanly:")
print("  * tokens -- the bad set dumps 60 records per call; this should be the")
print("    most reliable difference, and it is pure Section 4.")
print("  * first tool -- with 19 vague overlapping names, picking 'search' or")
print("    'find' over 'dump_tasks' is Section 9's semantic confusion happening")
print("    in front of you. The good set's namespacing leaves one obvious choice.")
print("  * correct -- the bad set must compare 60 due-dates in-context; the good")
print("    set asks the server via status='overdue'. This is where an 8B model")
print("    most often slips, either missing a task or inventing one.")
print()
print("If the BAD set still ties or wins: report that honestly rather than")
print("re-running until it agrees. A 60-record store on a 20-tool library is a")
print("mild version of the conditions Section 9 describes -- the cited research")
print("sees the sharp cliff at 50+ tools with real overlap. The mechanism is")
print("what matters, and the first-tool column shows it even when the final")
print("answer happens to survive.")

Question: Which of sam's tasks are overdue? A task is overdue if it is still open and its due date is before 2026-08-01. List their titles. Use the tools -- do not guess.
Correct answer (10 task(s)): ['Migrate database #3', 'Interview notes #11', 'Migrate database #15', 'Interview notes #23', 'Migrate database #27', 'Interview notes #35', 'Migrate database #39', 'Interview notes #47', 'Migrate database #51', 'Interview notes #59']
Model: qwen3:8b  |  3 attempt(s) per tool set
Library size: BAD=19 tools, GOOD=19 tools

--------------------------------------------------------------------
BAD TOOL SET (19 tools)
--------------------------------------------------------------------
  attempt 1: correct=False steps=3 tool_errors=0 tokens=10093
    first tool chosen: search
    all tools called : ['search', 'dump_tasks']
    answer: Here’s a structured summary of the provided task data (assuming typical categories like priority, assignee, and project):  ---  ### **Total Tasks**   - **60 tasks

### What This Cell Is Actually Showing

The one-sentence version: **this is the only cell in the whole notebook where the two tool sets face a real model, at real scale, and the result is measured rather than hand-computed.**

#### What actually happens, step by step

The same real question -- "which of sam's tasks are overdue?" -- gets asked 3 times against the BAD 19-tool library and 3 times against the GOOD 19-tool library. Each attempt runs a small agent loop (`run_loop`) that lets the model call tools, feeds back results, and keeps going until the model gives a final text answer or hits a step limit. For every attempt, four things get recorded: whether the final answer was actually correct (`grade`), how many steps it took, how many tool calls errored, and -- the most direct test of the chapter's claim -- **which tool the model reached for first.**

`grade()` isn't just "did it mention the right tasks" -- it also fails an answer that names tasks that *aren't* actually overdue. That catches a specific, realistic failure: a model that tries to filter 60 records in its head and gets some of them wrong, which only the bad tool set forces it to attempt in the first place.

#### Why "first tool chosen" is the sharpest signal here

Final-answer correctness can be right for the wrong reason (a lucky guess) or wrong for a reason unrelated to tool design (arithmetic slip on a date). Which tool the model reaches for *first*, though, is a direct readout of whether the schema's name and description did their job the moment they mattered most -- before any tool result existed to lean on. With 19 vague, overlapping bad names, watching the model land on `search` or `find` instead of the actually-correct-but-generically-named tool is Section 9's failure mode happening in real time, not simulated.

#### How to read the summary table

```
  tool set    correct   avg steps   avg errors   avg tokens
  BAD          x/3         ...          ...          ...
  GOOD         x/3         ...          ...          ...
```

Read `correct` and `first tool` together, not separately: a tool set can accidentally get the right final answer while still reaching for a confusing tool first, or vice versa. The token column should track Part 1 of the previous cell (bad results are bigger) rather than Part 2 (schema declaration size), since here it's actual usage being measured, not just the up-front declaration.

#### The honesty check built into this cell

An 8B local model is noisy, and 3 attempts per tool set is a small sample -- the code says this outright rather than hiding it. If the BAD set ties or even wins on a given run, that's a real result to report, not a sign to keep re-running until the numbers cooperate. This setup is a *mild* version of Section 9's conditions -- the research it's based on describes a much sharper cliff at 50+ tools with heavier overlap than 19 tools can reproduce. What should hold up even in a noisy run is the *mechanism* (the first-tool column showing confusion) more than the exact win/loss score.


## Structured Output: Constraining the Final Answer

Sections 2 through 7 were all about tool *calls* -- what the model sends. Section
6 is about the other direction: forcing the model's **final answer** into a
schema, so the code parsing it never has to handle "almost-JSON wrapped in a
sentence."

The distinction the notes insist on is worth restating, because targeting the
wrong knob is a genuinely confusing bug: `response_format` constrains the
model's *prose response*. It does **not** constrain tool arguments. For those you
want `strict: true` on the tool's own schema. Using one where you meant the other
gives you no guarantee and a debugging session.

Two Ollama-specific truths for this cell, both worth knowing before you rely on
it in your own code:

1. Ollama's OpenAI-compatible endpoint **does** accept
   `response_format={"type": "json_schema", ...}`, but its support is less
   complete than real OpenAI's -- some JSON Schema features (notably string
   `format` values like `date-time`) are ignored. Ollama's own native
   `/api/chat` endpoint takes a bare schema in a `format` field instead.
2. There are open reports of **tool calling being ignored when
   `response_format` is present** on self-hosted Ollama. That is a good reason
   to keep this call tool-free -- which matches Section 6's own advice anyway:
   structured output is for extraction, not for actions.

In [14]:
# A strict-mode-legal schema, following all three of Section 6's rules:
#   1. additionalProperties: false on every object
#   2. every field listed in "required"
#   3. optional fields expressed as a null union, not omitted from required
TASK_SUMMARY_SCHEMA = {
    "type": "object",
    "properties": {
        "assignee": {"type": "string"},
        "open_task_count": {"type": "integer"},
        "highest_priority": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
        "note": {"anyOf": [{"type": "string"}, {"type": "null"}]},
    },
    "required": ["assignee", "open_task_count", "highest_priority", "note"],
    "additionalProperties": False,
}

# The facts are handed over in the prompt -- no tools, by design (see the note
# above about tool calling and response_format on self-hosted Ollama).
alex_open = [t for t in TASK_STORE if t["assignee"] == "alex" and t["status"] == "open"]
facts = "\n".join(f"- {t['title']} (priority {t['priority']}, due {t['due_date']})" for t in alex_open)

print("Facts given to the model (no tools involved):")
print(facts)
print()

try:
    response = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": "You extract structured data. Reply with JSON only."},
            {"role": "user", "content":
                f"Summarise alex's open tasks.\n{facts}\n"
                f"Return assignee, open_task_count, highest_priority, and note "
                f"(a one-sentence remark, or null)."},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "task_summary", "strict": True, "schema": TASK_SUMMARY_SCHEMA},
        },
    )
    raw = response.choices[0].message.content
    print(f"Raw model output: {raw!r}")
    print()

    parsed = json.loads(raw)   # with constrained decoding this cannot throw
    print("Parsed successfully:")
    for key, value in parsed.items():
        print(f"  {key:20s} = {value!r}")

    # Constrained decoding guarantees SHAPE, never CORRECTNESS. Check both.
    shape_ok = set(parsed) == set(TASK_SUMMARY_SCHEMA["required"])
    enum_ok = parsed.get("highest_priority") in ["low", "medium", "high", "critical"]
    print()
    print(f"  shape matches schema exactly: {shape_ok}")
    print(f"  highest_priority within enum: {enum_ok}")

    truth_count = len(alex_open)
    truth_priority = "critical" if any(t["priority"] == "critical" for t in alex_open) else "high"
    print()
    print(f"  ground truth: open_task_count={truth_count}, highest_priority={truth_priority!r}")
    print(f"  model said:   open_task_count={parsed.get('open_task_count')!r}, "
          f"highest_priority={parsed.get('highest_priority')!r}")
    if parsed.get("open_task_count") != truth_count:
        print()
        print("  ^ Worth noticing if these disagree: the schema was still satisfied.")
        print("    Constrained decoding makes malformed output impossible; it does")
        print("    nothing whatsoever about wrong output. That gap is Chapter 6's job.")

except json.JSONDecodeError as exc:
    print(f"json.loads FAILED: {exc}")
    print("On real OpenAI with strict:true this is structurally impossible.")
    print("Seeing it here means your Ollama build ignored the json_schema and fell")
    print("back to free-form text -- try Ollama's native /api/chat 'format' field,")
    print("or upgrade Ollama.")
except Exception as exc:
    print(f"Request failed: {type(exc).__name__}: {exc}")
    print("(is 'ollama serve' running? does this model support structured outputs?)")

Facts given to the model (no tools involved):
- Fix login bug (priority high, due 2026-07-15)
- Refactor auth module (priority high, due 2026-07-20)
- Fix payment bug (priority critical, due 2026-07-01)

Raw model output: '{\n  "assignee": "Alex",\n  "open_task_count": 3,\n  "highest_priority": "critical",\n  "note": null\n}'

Parsed successfully:
  assignee             = 'Alex'
  open_task_count      = 3
  highest_priority     = 'critical'
  note                 = None

  shape matches schema exactly: True
  highest_priority within enum: True

  ground truth: open_task_count=3, highest_priority='critical'
  model said:   open_task_count=3, highest_priority='critical'


### What This Cell Is Actually Showing

The one-sentence version: **constraining a model's output to match a JSON schema guarantees the shape will be valid -- it says absolutely nothing about whether the values inside it are true.**

#### The two different knobs, and why mixing them up wastes an afternoon

There are two separate things you can constrain, and they live in different places:

- `response_format={"type": "json_schema", ...}` constrains the model's **final written answer** -- the prose reply it gives you.
- `strict: true` on a **tool's own schema** constrains the **arguments of a tool call** -- what the model sends *to* a function.

They look similar (both are "a schema the model must obey") but they apply to completely different parts of a conversation. If your tool calls are coming out malformed, adding `response_format` won't fix it -- that's not the knob that touches tool arguments at all.

#### What this specific cell does

It hands the model plain facts about alex's open tasks directly in the prompt text -- no tools involved on purpose, partly because that's what Section 6 recommends (structured output is for extracting/summarizing information, not for taking actions) and partly because of a real Ollama quirk: some self-hosted setups have been seen to silently ignore tool calls when `response_format` is also present in the same request. Keeping this call tool-free sidesteps that entirely.

It then asks for a summary matching `TASK_SUMMARY_SCHEMA` -- `assignee`, `open_task_count`, `highest_priority`, and `note`. Because the schema is enforced via constrained decoding, `json.loads(raw)` on the model's reply is guaranteed not to throw: the model is not free to produce a stray sentence before the JSON, a missing field, or a value outside the `priority` enum. That's a real, useful guarantee -- most of the pain of "parsing whatever a model wrote" disappears.

#### The gap this cell deliberately exposes

Look at the very end of the cell: it compares the model's `open_task_count` and `highest_priority` against a hand-computed ground truth from the same data. The schema being satisfied says nothing about whether those numbers are *right* -- a model can produce perfectly valid, schema-legal JSON that simply miscounts alex's tasks or picks the wrong priority. Constrained decoding is a guarantee about **shape**, never about **truth**. That gap -- catching a value that's wrong despite being well-formed -- is a different problem the notes point to Chapter 6 for, not something this schema (or any schema) can close on its own.


## Key Takeaways

You built two tool sets over identical data and measured the difference in the
token bill, the model's first tool choice, and whether it got the answer right at
all. Nothing about the underlying capability changed between them.

The most useful thing in this notebook may be the part that failed first. The
original comparison used 3 tools over 12 records and found *no difference*, which
is exactly the result you would use to conclude that this whole chapter is
overthinking things. It was a measurement problem: none of the conditions
Section 9 names were present. Designing an experiment that *can* fail -- enough
tools, enough overlap, enough data that filtering matters -- is inseparable from
the claim being worth anything, and this applies well beyond tool design.

What each exercise was actually teaching:

1. **The three rewrites** -- the schema is the whole specification. Naming,
   description length, and flat arguments are the entire interface, and in
   OpenAI format they live inside `function.parameters` rather than
   `input_schema`, while the design principles themselves transfer unchanged.
2. **`search_logs_paginated`** -- truncation is fine; *silent* truncation is a
   lie the model has no way to detect, and one `message` field is the whole fix.
3. **`validate_tool_input`** -- an error message is read by a model with one
   retry. Name the field, echo the bad value, show a correct one. On
   OpenAI/Ollama, where there is no `is_error` flag, that text is the only
   signal you get.
4. **`run_tool_with_gate`** -- `destructive: true` is inert until code reads it
   *before* dispatch. That flag is the hook Chapter 16's permission layer needs
   to find, and this is what it looks like when it actually holds.

And the two things this chapter deliberately could not fix, both visible in the
last section's output: constrained decoding guaranteed the *shape* of that final
JSON and said nothing about whether the numbers in it were true, and no amount of
schema polish decided *when* the model should stop and check its own work. Those
are Chapter 6's (verification) and Chapter 4's (context management) problems.

**Connection forward:** Chapter 4 takes the token arithmetic above and
generalises it past tool schemas into the whole context window -- system prompt,
tool definitions, memory, and the growing transcript tail -- as a single finite
budget with four operations (write, select, compress, isolate) available for
managing it.